In [ ]:
#-- Load libraries
library(readr)
library(dplyr)
library(data.table)
library(ggplot2)
library(lubridate)
library(purrr)
library(stringr)
library(tidyverse)
library(openxlsx)
library(nnet)
library(tidyr)

In [ ]:
#system(paste0("gsutil cp ", Sys.getenv("WORKSPACE_BUCKET"), "/results/Phenotypes.csv /home/jupyter/workspaces/cypphenoconversion/data/Phenotypes.csv"))

#-- Load Phenotypes
pheno <- read.csv("/home/jupyter/workspaces/cypphenoconversion/data/Phenotypes.csv")
#-- Set reference levels
pheno$Education_level = relevel(as.factor(pheno$Education_level), ref = "Less than a high school degree")
pheno <- pheno %>% 
    mutate(Income_quantitative =
          case_when(Income == ">= 200k" ~ 8,
                   Income == "150-200k" ~ 7,
                   Income == "100-150k" ~ 6,
                   Income == "75-100k" ~ 5,
                   Income == "50-75k" ~ 4,
                   Income == "35-50k" ~ 3,
                   Income == "25-35k" ~ 2,
                   Income == "10-25k" ~ 1,
                   TRUE ~ NA_integer_))

In [ ]:
#system(paste0("gsutil cp ", Sys.getenv("WORKSPACE_BUCKET"), "/results/Main_CYP_Outcomes_90days_Phenoconversion_noAD_2scripts_S_C.csv /home/jupyter/workspaces/cypphenoconversion/data/Main_CYP_Outcomes_90days_Phenoconversion_noAD_2scripts_S_C.csv"))

#-- Load outcomes
outcome <- read.csv("/home/jupyter/workspaces/cypphenoconversion/data/Main_CYP_Outcomes_90days_Phenoconversion_noAD_2scripts_S_C.csv") %>%
  mutate(
    FirstDispense.treatment = as.Date(FirstDispense.treatment),
    ExpectedEndDate.treatment = as.Date(ExpectedEndDate.treatment),
    FirstDispense.treatment.episode = as.Date(FirstDispense.treatment.episode),
    ExpectedEndDate.treatment.episode = as.Date(ExpectedEndDate.treatment.episode),
    FirstDispense.other.episode = as.Date(FirstDispense.other.episode),
    ExpectedEndDate.other.episode = as.Date(ExpectedEndDate.other.episode),
    Incidence = as.Date(Incidence),
    DateFirstDispense = as.Date(DateFirstDispense)
  )

outcome <- outcome %>%
  mutate(
    has_any_AD_modulator_ever = case_when(
      has_AD_cyp2d6_modulator_any_followup == 1L | 
      has_AD_cyp2c19_modulator_any_followup == 1L ~ 1L,
      TRUE ~ 0L
    )
  )

In [ ]:
#-- Read in predicted ancestries
library(readr)
workspace_bucket <- Sys.getenv("WORKSPACE_BUCKET")
anc_file_path <- file.path(workspace_bucket, "ancestry/ancestry_preds.tsv")
anc_df <- read_tsv(pipe(sprintf("gsutil cat %s", anc_file_path)))
anc_df <- anc_df %>% select(research_id, ancestry_pred)

outcome <- outcome %>%
    left_join(anc_df, by = c("person_id" = "research_id")) %>%
    mutate(
    ancestry_pred = case_when(
        ancestry_pred %in% c("sas", "eas", "mid") ~ "Other",
        !is.na(ancestry_pred) ~ ancestry_pred,
        TRUE ~ ancestry_pred))

ancestries <- unique(outcome$ancestry_pred)
table(outcome$ancestry_pred)

In [ ]:
#== Identify drugs with at least 1000 people
drugs <- outcome %>%
    group_by(drug.treatment) %>%
    summarise(
        count = n_distinct(person_id)
    ) %>%
    filter(count >= 1000) %>%
    pull(drug.treatment)
print(drugs)


In [ ]:
#== Helper: drop sparse binary predictors from a formula
drop_sparse_binary_terms <- function(formula, data, outcome_var = "status", min_cell = 2) {
  term_labels <- attr(terms(formula), "term.labels")
  
  terms_to_drop <- c()
  
  for (term in term_labels) {
    if (!term %in% names(data)) next
    col <- data[[term]]
    
    # Only check binary 0/1 variables (modulators), not factors or continuous
    if (!is.numeric(col) && !is.integer(col)) next
    unique_vals <- unique(col[!is.na(col)])
    if (!all(unique_vals %in% c(0, 1))) next
    
    # Cross-tab exposed (==1) against outcome
    exposed <- data[col == 1 & !is.na(col), ]
    n_exposed_cases    <- sum(exposed[[outcome_var]] == 1, na.rm = TRUE)
    n_exposed_controls <- sum(exposed[[outcome_var]] == 0, na.rm = TRUE)
    
    if (n_exposed_cases < min_cell || n_exposed_controls < min_cell) {
      message(sprintf("    Dropping sparse term '%s': exposed cases=%d, exposed controls=%d",
                      term, n_exposed_cases, n_exposed_controls))
      terms_to_drop <- c(terms_to_drop, term)
    }
  }
  
  if (length(terms_to_drop) > 0) {
    formula <- stats::update(formula,
      paste(". ~ . -", paste(terms_to_drop, collapse = " - ")))
  }
  
  formula
}

In [ ]:
# Define parameters
predictors <- c("CYP2C19", "CYP2D6", "CYP2B6")
ancestry_groups <- c("Full", "European")
cyp_windows <- c("any_followup", "baseline_only", "concom_only")

drug_results_list <- list()

for (drug in drugs) {
  
  print(paste("Processing drug:", drug))

  
  #-- First switch (first incidence of switch)
    first_switch <- outcome %>%
      filter(drug.treatment == drug) %>%
      filter(FinalClassification.treatment == "Switching") %>%
      group_by(person_id) %>%
      filter(Incidence == min(Incidence)) %>%
      filter(FirstDispense.treatment.episode == min(FirstDispense.treatment.episode)) %>%
      filter(ExpectedEndDate.treatment.episode == max(ExpectedEndDate.treatment.episode)) %>%
      filter(ExpectedEndDate.treatment == min(ExpectedEndDate.treatment)) %>%
      ungroup()

    #-- No switch (first incidence of continuation among non-switchers)
    no_switch <- outcome %>%
      filter(drug.treatment == drug) %>%
      filter(!person_id %in% first_switch$person_id) %>%
      filter(FinalClassification.treatment == "Continuation") %>%
      group_by(person_id) %>%
      arrange(Incidence, FirstDispense.treatment) %>%
      slice(1) %>%
      ungroup()

  #== Input data for switching vs. non-switching
  dat <- rbind(first_switch, no_switch)
    
    #==== Find number of other AD dispensed prior index date of drug
    first_switch_other <- dat %>%
        left_join(outcome, by = c("person_id"), suffix = c(".first", ".second")) %>%
        filter(FirstDispense.treatment.episode.second < Index.first) %>%
        filter(drug.treatment.first != drug.treatment.second)
    count <- first_switch_other %>%
        group_by(person_id) %>%
        summarise(
        num_prior_AD = length(unique(drug.treatment.second)))
    dat <- dat %>%
    left_join(count, by = "person_id") %>%
        mutate(num_prior_AD = ifelse(is.na(num_prior_AD), 0, num_prior_AD))
  
  #== Calculate age at first AD dispense
  input_data <- dat %>%
    left_join(pheno, by = "person_id") %>%
    mutate(
      age_at_first_AD = interval(date_of_birth, DateFirstDispense) %/% years(1)
    )
    
  #== Create binary outcome variable
  input_data <- input_data %>%
    mutate(
      status = case_when(
        FinalClassification.treatment == "Switching" ~ 1L,
        FinalClassification.treatment != "Switching" ~ 0L,
        TRUE ~ NA_integer_
      )
    )
    
  #== Collapse CYP2B6 intermediate and Poor into one category
  input_data <- input_data %>%
    mutate(
        CYP2B6 = case_when(
        CYP2B6 %in% c("Poor", "Intermediate") ~ "Poor/Intermediate",
        TRUE ~ CYP2B6)
        )
  
  # Set raw phenotypes as factors (do this once before the loops)
  input_data$CYP2C19 <- factor(
    input_data$CYP2C19,
    levels = c("Poor", "Intermediate", "Normal", "Rapid", "Ultrarapid")
  )
  input_data$CYP2C19 <- stats::relevel(input_data$CYP2C19, ref = "Normal")
  
  input_data$CYP2D6 <- factor(
    input_data$CYP2D6,
    levels = c("Poor", "Intermediate", "Normal", "Ultrarapid")
  )
  input_data$CYP2D6 <- stats::relevel(input_data$CYP2D6, ref = "Normal")
  
  input_data$CYP2B6 <- factor(
    input_data$CYP2B6,
    levels = c("Poor/Intermediate", "Normal", "Ultrarapid")
  )
  input_data$CYP2B6 <- stats::relevel(input_data$CYP2B6, ref = "Normal")
  
  #==========================
  # Loop through conditions
  #==========================
  
  all_results <- list()
  
  for (window in cyp_windows) {
    for (ancestry in ancestry_groups) {
      
      message(sprintf("\n>>> Drug: %s, Window: %s, Ancestry: %s",
                      drug, window, ancestry))
      
      #== Filter ancestry
      if (ancestry == "European") {
        input_data_pop <- input_data %>%
          filter(ancestry_pred == "eur")
      } else {
        input_data_pop <- input_data
      }
      
      #== Map window-specific CYP columns into generic names
      c19_adj_col <- paste0("CYP2C19_", window, "_phenoconverted")
      d6_adj_col  <- paste0("CYP2D6_", window, "_phenoconverted")
      b6_adj_col  <- paste0("CYP2B6_", window, "_phenoconverted")
      
      # CYP2C19 modulators
      c19_s_inhib_col <- paste0(window, "_cyp2c19_strong_inhibitors")
      c19_m_inhib_col <- paste0(window, "_cyp2c19_moderate_inhibitors")
      c19_w_inhib_col <- paste0(window, "_cyp2c19_weak_inhibitors")
      c19_m_ind_col   <- paste0(window, "_cyp2c19_moderate_inducers")
      c19_w_ind_col   <- paste0(window, "_cyp2c19_weak_inducers")
      
      # CYP2D6 modulators
      d6_s_inhib_col <- paste0(window, "_cyp2d6_strong_inhibitors")
      d6_m_inhib_col <- paste0(window, "_cyp2d6_moderate_inhibitors")
      d6_w_inhib_col <- paste0(window, "_cyp2d6_weak_inhibitors")
      
      # CYP2B6 modulators
      b6_s_inhib_col <- paste0(window, "_cyp2b6_strong_inhibitors")
      b6_m_inhib_col <- paste0(window, "_cyp2b6_moderate_inhibitors")
      b6_w_inhib_col <- paste0(window, "_cyp2b6_weak_inhibitors")
      b6_m_ind_col   <- paste0(window, "_cyp2b6_moderate_inducers")
      b6_w_ind_col   <- paste0(window, "_cyp2b6_weak_inducers")
      
      # CYP neg controls
      prav_col <- paste0(window, "_pravastatin_neg_cont")
      rosu_col <- paste0(window, "_rosuvastatin_neg_cont")
        
      # AD modulator indicators (already exist in data)
      has_AD_d6_col  <- paste0("has_AD_cyp2d6_modulator_", window)
      has_AD_c19_col <- paste0("has_AD_cyp2c19_modulator_", window)
      
      #== Create window-specific adjusted phenotypes and modulators
      input_data_window <- input_data_pop %>%
        mutate(
          # Phenoconverted phenotypes
          CYP2C19_adjusted = .data[[c19_adj_col]],
          CYP2D6_adjusted  = .data[[d6_adj_col]],
          CYP2B6_adjusted  = .data[[b6_adj_col]],
          
          # CYP2C19 modulators
          cyp2c19_strong_inhib = .data[[c19_s_inhib_col]],
          cyp2c19_mod_inhib    = .data[[c19_m_inhib_col]],
          cyp2c19_weak_inhib   = .data[[c19_w_inhib_col]],
          cyp2c19_mod_ind      = .data[[c19_m_ind_col]],
          cyp2c19_weak_ind     = .data[[c19_w_ind_col]],
          
          # CYP2D6 modulators
          cyp2d6_strong_inhib = .data[[d6_s_inhib_col]],
          cyp2d6_mod_inhib    = .data[[d6_m_inhib_col]],
          cyp2d6_weak_inhib   = .data[[d6_w_inhib_col]],
          
          # CYP2B6 modulators
          cyp2b6_strong_inhib = .data[[b6_s_inhib_col]],
          cyp2b6_mod_inhib    = .data[[b6_m_inhib_col]],
          cyp2b6_weak_inhib   = .data[[b6_w_inhib_col]],
          cyp2b6_mod_ind      = .data[[b6_m_ind_col]],
          cyp2b6_weak_ind     = .data[[b6_w_ind_col]],
          
          # CYP negative controls
          prav_neg_cont        = .data[[prav_col]],
          rosu_neg_cont        = .data[[rosu_col]],
            
          ## NEW: Create "any inhibitor" variables
          cyp2d6_any_inhib = as.integer(cyp2d6_strong_inhib == 1 | 
                                       cyp2d6_mod_inhib == 1 | 
                                       cyp2d6_weak_inhib == 1),

          cyp2c19_any_inhib = as.integer(cyp2c19_strong_inhib == 1 | 
                                        cyp2c19_mod_inhib == 1 | 
                                        cyp2c19_weak_inhib == 1),

          cyp2b6_any_inhib = as.integer(cyp2b6_strong_inhib == 1 | 
                                       cyp2b6_mod_inhib == 1 | 
                                       cyp2b6_weak_inhib == 1),

          cyp2c19_any_ind = as.integer(cyp2c19_mod_ind == 1 | 
                                      cyp2c19_weak_ind == 1),

          cyp2b6_any_ind = as.integer(cyp2b6_mod_ind == 1 | 
                                     cyp2b6_weak_ind == 1),
            
          # AD modulator indicators
          has_AD_cyp2d6_modulator  = .data[[has_AD_d6_col]],
          has_AD_cyp2c19_modulator = .data[[has_AD_c19_col]],
          has_any_AD_modulator = as.integer(
            has_AD_cyp2d6_modulator == 1L | has_AD_cyp2c19_modulator == 1L
          )
        
        )
        
    #-- Combine CYP2B6 strong and moderate inhibitors and combine CYP2B6 poor and intermediate
    input_data_window <- input_data_window %>%
        mutate(
            cyp2b6_mod_strong_inhib = case_when(
                (cyp2b6_mod_inhib == 1 | cyp2b6_strong_inhib == 1) ~ 1,
                TRUE ~ 0),
            cyp2b6_weak_mod_ind = case_when(
                (cyp2b6_mod_ind == 1 | cyp2b6_weak_ind == 1) ~ 1,
                TRUE ~ 0),
            cyp2c19_weak_mod_ind = case_when(
                cyp2c19_weak_ind == 1 | cyp2c19_mod_ind == 1 ~ 1,
                TRUE ~ 0),
            cyp2d6_mod_strong_inhib = case_when(
                cyp2d6_mod_inhib == 1 | cyp2d6_strong_inhib == 1 ~ 1,
                TRUE ~ 0)
            ) %>%
        mutate(
            CYP2B6_adjusted = case_when(
                CYP2B6_adjusted %in% c("Poor", "Intermediate") ~ "Poor/Intermediate",
                TRUE ~ CYP2B6_adjusted)
        )
        
    ## ADD THIS DIAGNOSTIC RIGHT AFTER:
    message("\nChecking created columns:")
    message("  CYP2D6_adjusted NAs: ", sum(is.na(input_data_window$CYP2D6_adjusted)), " / ", nrow(input_data_window))
    message("  CYP2C19_adjusted NAs: ", sum(is.na(input_data_window$CYP2C19_adjusted)), " / ", nrow(input_data_window))
    message("  CYP2B6_adjusted NAs: ", sum(is.na(input_data_window$CYP2B6_adjusted)), " / ", nrow(input_data_window))

    message("\nChecking if source columns exist:")
    message("  ", c19_adj_col, " exists? ", c19_adj_col %in% names(input_data_pop))
    message("  ", d6_adj_col, " exists? ", d6_adj_col %in% names(input_data_pop))
    message("  ", b6_adj_col, " exists? ", b6_adj_col %in% names(input_data_pop))

      
      #== Set adjusted phenotypes as factors
      input_data_window$CYP2C19_adjusted <- factor(
        input_data_window$CYP2C19_adjusted,
        levels = c("Poor", "Intermediate", "Normal", "Rapid", "Ultrarapid")
      )
      input_data_window$CYP2C19_adjusted <- stats::relevel(
        input_data_window$CYP2C19_adjusted,
        ref = "Normal"
      )
      
      input_data_window$CYP2D6_adjusted <- factor(
        input_data_window$CYP2D6_adjusted,
        levels = c("Poor", "Intermediate", "Normal", "Ultrarapid")
      )
      input_data_window$CYP2D6_adjusted <- stats::relevel(
        input_data_window$CYP2D6_adjusted,
        ref = "Normal"
      )
      
      input_data_window$CYP2B6_adjusted <- factor(
        input_data_window$CYP2B6_adjusted,
        levels = c("Poor/Intermediate", "Normal", "Ultrarapid")
      )
      input_data_window$CYP2B6_adjusted <- stats::relevel(
        input_data_window$CYP2B6_adjusted,
        ref = "Normal"
      )
    
      input_data$ancestry_pred <- factor(input_data$ancestry_pred)
      input_data$ancestry_pred <- relevel(input_data$ancestry_pred, ref = "eur")
    
      
      #== Build complete-case dataset
        common_vars <- c(
          "person_id", "status",
          "age_at_first_AD", "sex_at_birth", "Education_level", "Income_quantitative",
          # modulators by strength
          "cyp2d6_strong_inhib", "cyp2d6_mod_inhib", "cyp2d6_weak_inhib",
          "cyp2c19_strong_inhib", "cyp2c19_mod_inhib", "cyp2c19_weak_inhib",
          "cyp2c19_weak_mod_ind", "num_prior_AD",
          "cyp2b6_mod_strong_inhib", "cyp2b6_weak_inhib",
          "cyp2b6_weak_mod_ind", "cyp2d6_mod_strong_inhib",
          # NEW: any strength inhibitors
          "cyp2d6_any_inhib", "cyp2c19_any_inhib", "cyp2b6_any_inhib",
          "cyp2c19_any_ind", "cyp2b6_any_ind",
          # CYP negative controls
          "prav_neg_cont", "rosu_neg_cont",
          # raw phenotypes
          "CYP2D6", "CYP2C19", "CYP2B6",
          # adjusted phenotypes
          "CYP2D6_adjusted", "CYP2C19_adjusted", "CYP2B6_adjusted",
         # AD CYP Modulators
         "has_AD_cyp2d6_modulator", "has_AD_cyp2c19_modulator", "has_any_AD_modulator", "has_any_AD_modulator_ever"
        )
      
      if (ancestry == "European") {
        base_dat <- input_data_window %>%
          select(all_of(common_vars)) %>%
          filter(complete.cases(.))
      } else {
        base_dat <- input_data_window %>%
          select(all_of(c(common_vars, "ancestry_pred"))) %>%
          filter(complete.cases(.))
      }
      
      base_dat <- base_dat %>%
          mutate(Education_level = factor(Education_level))
        
      # Skip if no data
      if (nrow(base_dat) == 0) {
        message("  Skipping: No data after complete.cases")
        next
      }
      
      #== Create subset excluding AD modulators
      no_AD_mod_dat <- base_dat %>%
        filter(has_any_AD_modulator_ever == 0L)
        
    #== Create subset excluding ALL modulators (any CYP inhibitor/inducer)
    no_any_mod_dat <- base_dat %>%
      filter(cyp2d6_any_inhib == 0 &
             cyp2c19_any_inhib == 0 &
             cyp2c19_any_ind == 0 &
             cyp2b6_any_inhib == 0 &
             cyp2b6_any_ind == 0)

    # Calculate sample sizes
    no_any_mod_counts <- no_any_mod_dat %>%
      group_by(status) %>%
      summarise(n = n(), .groups = "drop")

    no_any_n_cases    <- if(1 %in% no_any_mod_counts$status) no_any_mod_counts$n[no_any_mod_counts$status == 1] else 0
    no_any_n_controls <- if(0 %in% no_any_mod_counts$status) no_any_mod_counts$n[no_any_mod_counts$status == 0] else 0
    no_any_n_totals   <- no_any_n_cases + no_any_n_controls

    message(sprintf("  No-any-modulator subset: N=%d (Cases=%d, Controls=%d)", 
                    no_any_n_totals, no_any_n_cases, no_any_n_controls))
      
      # Calculate sample sizes for no-AD-modulator subset
      no_AD_switch_counts <- no_AD_mod_dat %>%
        group_by(status) %>%
        summarise(n = n(), .groups = "drop")
      
      no_AD_n_cases    <- if(1 %in% no_AD_switch_counts$status) no_AD_switch_counts$n[no_AD_switch_counts$status == 1] else 0
      no_AD_n_controls <- if(0 %in% no_AD_switch_counts$status) no_AD_switch_counts$n[no_AD_switch_counts$status == 0] else 0
      no_AD_n_totals   <- no_AD_n_cases + no_AD_n_controls
      
      message(sprintf("  No-AD-modulator subset: N=%d (Cases=%d, Controls=%d)", 
                      no_AD_n_totals, no_AD_n_cases, no_AD_n_controls)) 
        
      #== Overall switch prevalence
      switch_counts <- base_dat %>%
        group_by(status) %>%
        summarise(n = n(), .groups = "drop")
      
      n_cases    <- switch_counts$n[switch_counts$status == 1]
      n_controls <- switch_counts$n[switch_counts$status == 0]
      
      if (length(n_cases) == 0 || length(n_controls) == 0) {
        message("  Skipping: Missing cases or controls")
        next
      }
      
      n_totals <- n_cases + n_controls
      

    # Require minimum sample size
    if (n_cases < 20 || n_controls < 20 || n_totals < 1000) {
      message(sprintf("  Skipping: Insufficient sample (N=%d, Cases=%d, Controls=%d)",
                      n_totals, n_cases, n_controls))
      next
        }
      
      message(sprintf("  N=%d (Cases=%d, Controls=%d)", n_totals, n_cases, n_controls))
        
    # Calculate sample sizes for no-AD-modulator subset
    no_AD_switch_counts <- no_AD_mod_dat %>%
      group_by(status) %>%
      summarise(n = n(), .groups = "drop")

    no_AD_n_cases    <- if(1 %in% no_AD_switch_counts$status) no_AD_switch_counts$n[no_AD_switch_counts$status == 1] else 0
    no_AD_n_controls <- if(0 %in% no_AD_switch_counts$status) no_AD_switch_counts$n[no_AD_switch_counts$status == 0] else 0
    no_AD_n_totals   <- no_AD_n_cases + no_AD_n_controls

    message(sprintf("  No-AD-modulator subset: N=%d (Cases=%d, Controls=%d)", 
                    no_AD_n_totals, no_AD_n_cases, no_AD_n_controls))

    # Skip all models if no-AD-modulator subset is too small
    if (no_AD_n_cases < 20 || no_AD_n_controls < 20 || no_AD_n_totals < 1000) {
      message(sprintf("  Skipping all models: no-AD-modulator subset too small (N=%d, Cases=%d, Controls=%d)",
                      no_AD_n_totals, no_AD_n_cases, no_AD_n_controls))
      next
    }
      
      results_list <- list()
      
      #== Helper function to fit GLM
      fit_glm_and_tidy <- function(glm_formula, model_name, independent_label,
                                  data = base_dat, 
                                   n_total = n_totals, 
                                   n_case = n_cases, 
                                   n_control = n_controls) {
        dat <- droplevels(data)
        
        # Check outcome has both 0 and 1
        if (dplyr::n_distinct(dat$status, na.rm = TRUE) < 2) {
          message("  Skipping model ", model_name, ": outcome has <2 levels")
          return(NULL)
        }
        
        # Get predictor term labels
        term_labels <- attr(terms(glm_formula), "term.labels")
        
        # Find factor predictors with <2 levels
        bad_terms <- term_labels[
          sapply(term_labels, function(v) {
            if (v %in% names(dat)) {
              is.factor(dat[[v]]) && dplyr::n_distinct(dat[[v]], na.rm = TRUE) < 2
            } else {
              FALSE
            }
          })
        ]
        
        if (length(bad_terms) > 0) {
          message("  Dropping factor(s) with <2 levels: ",
                  paste(bad_terms, collapse = ", "))
          glm_formula <- stats::update(glm_formula,
                                       paste(". ~ . -", paste(bad_terms, collapse = " - ")))
        }
          
        # ---- EXPLICIT EDUCATION LEVEL SPARSE CHECK ----
        if ("Education_level" %in% names(dat)) {
          dat$Education_level <- factor(dat$Education_level)

          tbl_ed <- table(dat$Education_level, dat$status, useNA = "no")

          if ("0" %in% colnames(tbl_ed) && "1" %in% colnames(tbl_ed)) {
            sparse_ed <- rownames(tbl_ed)[
              tbl_ed[, "1"] < 2 | tbl_ed[, "0"] < 2
            ]

            if (length(sparse_ed) > 0) {
              # Find a non-sparse level to use as reference
              non_sparse <- setdiff(levels(dat$Education_level), sparse_ed)
              if (length(non_sparse) > 0) {
                dat$Education_level <- relevel(dat$Education_level, ref = non_sparse[1])
              }
              message("  Removing sparse Education_level categories: ",
                      paste(sparse_ed, collapse = ", "))
              dat <- dat[!as.character(dat$Education_level) %in% sparse_ed, ]
              dat$Education_level <- droplevels(dat$Education_level)
            }
          }
        }


      # ---- NEW: DROP SPARSE BINARY MODULATOR TERMS ----
      glm_formula <- drop_sparse_binary_terms(glm_formula, data = dat, min_cell = 2)

        # Fit model
        glm_model <- tryCatch({
          stats::glm(
            glm_formula,
            data   = dat,
            family = stats::binomial(link = "logit")
          )
        }, error = function(e) {
          message("  Error fitting model ", model_name, ": ", e$message)
          return(NULL)
        })
        
        if (is.null(glm_model)) return(NULL)
        
        sm <- summary(glm_model)$coefficients
        
        coef_vec <- sm[, "Estimate"]
        se_vec   <- sm[, "Std. Error"]
        z_vec    <- sm[, "z value"]
        p_vec    <- sm[, "Pr(>|z|)"]
        
        lower_95 <- coef_vec - 1.96 * se_vec
        upper_95 <- coef_vec + 1.96 * se_vec
        
        glm_results <- data.frame(
          variable   = rownames(sm),
          coef       = coef_vec,
          odds_ratio = exp(coef_vec),
          se         = se_vec,
          z          = z_vec,
          p_value    = p_vec,
          lower_95   = exp(lower_95),
          upper_95   = exp(upper_95),
          stringsAsFactors = FALSE
        )
        rownames(glm_results) <- NULL
        
        glm_results$Model      <- model_name
        glm_results$Ancestry <- ancestry
        glm_results$Window     <- window
        glm_results$Drug       <- drug
        glm_results$N_Total    <- n_total
        glm_results$N_Cases    <- n_case
        glm_results$N_Controls <- n_control
        
        glm_results
      }
      
      #==========================
      # Model 1: Covariate-only (Exclude AD Modulators)
      #==========================
      if (ancestry == "European") {
        formula_covonly <- status ~ age_at_first_AD + sex_at_birth +
          Education_level + Income_quantitative + num_prior_AD +
          cyp2d6_mod_strong_inhib + cyp2d6_weak_inhib +
          cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
          cyp2c19_weak_mod_ind +
          cyp2b6_mod_strong_inhib + cyp2b6_weak_inhib +
          cyp2b6_weak_mod_ind + rosu_neg_cont + prav_neg_cont
      } else {
        formula_covonly <- status ~ age_at_first_AD + sex_at_birth +
          Education_level + Income_quantitative + ancestry_pred + num_prior_AD +
          cyp2d6_mod_strong_inhib + cyp2d6_weak_inhib +
          cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
          cyp2c19_weak_mod_ind + 
          cyp2b6_mod_strong_inhib + cyp2b6_weak_inhib +
          cyp2b6_weak_mod_ind + rosu_neg_cont + prav_neg_cont
      }
      
      results_list[["CovAll_NoCYP"]] <- fit_glm_and_tidy(
        glm_formula       = formula_covonly,
        model_name        = "CovAll_NoCYP",
        independent_label = "AllCovariates_NoCYP",
        data              = no_AD_mod_dat,
        n_total           = no_AD_n_totals,
        n_case            = no_AD_n_cases,
        n_control         = no_AD_n_controls
      )
      
      #==========================
      # Model 2d: Raw phenotypes (FULL SAMPLE - genotype only)
      #==========================
      if (ancestry == "European") {
        formula_cyp_main_raw <- status ~ age_at_first_AD + sex_at_birth +
          Education_level + Income_quantitative + num_prior_AD +
          CYP2D6 + CYP2C19 + CYP2B6
      } else {
        formula_cyp_main_raw <- status ~ age_at_first_AD + sex_at_birth +
          Education_level + Income_quantitative + ancestry_pred + num_prior_AD +
          CYP2D6 + CYP2C19 + CYP2B6
      }
      
      results_list[["CYP_Main_raw"]] <- fit_glm_and_tidy(
        glm_formula       = formula_cyp_main_raw,
        model_name        = "CYP_Main_raw",
        independent_label = "CYP2C19_CYP2D6_CYP2B6_raw",
        data              = base_dat,
        n_total           = n_totals,
        n_case            = n_cases,
        n_control         = n_controls
      )
        
      #==========================
      # Model 2a: Raw phenotypes + Modulators (Exclude AD Modulators)
      #==========================
      if (ancestry == "European") {
        formula_cyp_main_raw <- status ~ age_at_first_AD + sex_at_birth +
          Education_level + Income_quantitative + num_prior_AD +
          cyp2d6_mod_strong_inhib + cyp2d6_weak_inhib +
          cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
          cyp2c19_weak_mod_ind + 
          cyp2b6_mod_strong_inhib + cyp2b6_weak_inhib +
          cyp2b6_weak_mod_ind +
          CYP2D6 + CYP2C19 + CYP2B6
      } else {
        formula_cyp_main_raw <- status ~ age_at_first_AD + sex_at_birth +
          Education_level + Income_quantitative + ancestry_pred + num_prior_AD +
          cyp2d6_mod_strong_inhib + cyp2d6_weak_inhib +
          cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
          cyp2c19_weak_mod_ind + 
          cyp2b6_mod_strong_inhib + cyp2b6_weak_inhib +
          cyp2b6_weak_mod_ind + 
          CYP2D6 + CYP2C19 + CYP2B6
      }
      
      results_list[["CYP_Main_raw_plus_Mod"]] <- fit_glm_and_tidy(
        glm_formula       = formula_cyp_main_raw,
        model_name        = "CYP_Main_raw_plus_Mod",
        independent_label = "CYP2C19_CYP2D6_CYP2B6_raw_plus_Mod",
        data              = no_AD_mod_dat,
        n_total           = no_AD_n_totals,
        n_case            = no_AD_n_cases,
        n_control         = no_AD_n_controls
      )
      
      #==========================
      # Model 2b: Adjusted phenotypes (exclude AD Modulators)
      #==========================
      if (ancestry == "European") {
        formula_cyp_main_adj <- status ~ age_at_first_AD + sex_at_birth +
          Education_level + Income_quantitative + num_prior_AD +
          CYP2D6_adjusted + CYP2C19_adjusted + CYP2B6_adjusted
      } else {
        formula_cyp_main_adj <- status ~ age_at_first_AD + sex_at_birth +
          Education_level + Income_quantitative + ancestry_pred + num_prior_AD +
          CYP2D6_adjusted + CYP2C19_adjusted + CYP2B6_adjusted
      }
      
      results_list[["CYP_Main_adj"]] <- fit_glm_and_tidy(
        glm_formula       = formula_cyp_main_adj,
        model_name        = "CYP_Main_adj",
        independent_label = "CYP2C19_CYP2D6_CYP2B6_adj",
        data              = no_AD_mod_dat,
        n_total           = no_AD_n_totals,
        n_case            = no_AD_n_cases,
        n_control         = no_AD_n_controls
      )
        
        #====================================
        # Model 2c: Phenoconverted phenotpes + Modulators (exclude AD Modulators)
        #================================================
    if (ancestry == "European") {
        formula_cyp_main_adj_mod <- status ~ age_at_first_AD + sex_at_birth +
          Education_level + Income_quantitative + num_prior_AD +
          cyp2d6_mod_strong_inhib + cyp2d6_weak_inhib +
          cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
          cyp2c19_weak_mod_ind + 
          cyp2b6_mod_strong_inhib + cyp2b6_weak_inhib +
          cyp2b6_weak_mod_ind + 
          CYP2D6_adjusted + CYP2C19_adjusted + CYP2B6_adjusted
      } else {
        formula_cyp_main_adj_mod <- status ~ age_at_first_AD + sex_at_birth +
          Education_level + Income_quantitative + ancestry_pred + num_prior_AD +
          cyp2d6_mod_strong_inhib + cyp2d6_weak_inhib +
          cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
          cyp2c19_weak_mod_ind + 
          cyp2b6_mod_strong_inhib + cyp2b6_weak_inhib +
          cyp2b6_weak_mod_ind + 
          CYP2D6_adjusted + CYP2C19_adjusted + CYP2B6_adjusted
      }
      
      results_list[["CYP_Main_adj_plus_Mod"]] <- fit_glm_and_tidy(
        glm_formula       = formula_cyp_main_adj_mod,
        model_name        = "CYP_Main_adj_plus_Mod",
        independent_label = "CYP2C19_CYP2D6_CYP2B6_adj_mod",
        data              = no_AD_mod_dat,
        n_total           = no_AD_n_totals,
        n_case            = no_AD_n_cases,
        n_control         = no_AD_n_controls
      )
        
    #==========================
    # Model 3: Any strength inhibitors (exclude AD Modulators)
    #==========================
    if (ancestry == "European") {
      formula_any_inhib <- status ~ age_at_first_AD + sex_at_birth +
        Education_level + Income_quantitative + num_prior_AD +
        cyp2d6_any_inhib + 
        cyp2c19_any_inhib + cyp2c19_any_ind + rosu_neg_cont + prav_neg_cont +
        cyp2b6_any_inhib + cyp2b6_any_ind
    } else {
      formula_any_inhib <- status ~ age_at_first_AD + sex_at_birth +
        Education_level + Income_quantitative + ancestry_pred + num_prior_AD +
        cyp2d6_any_inhib + 
        cyp2c19_any_inhib + cyp2c19_any_ind + rosu_neg_cont + prav_neg_cont +
        cyp2b6_any_inhib + cyp2b6_any_ind
    }

    results_list[["Any_Inhib"]] <- fit_glm_and_tidy(
      glm_formula       = formula_any_inhib,
      model_name        = "Any_Inhib",
      independent_label = "AnyStrength_Inhibitors",
        data              = no_AD_mod_dat,
        n_total           = no_AD_n_totals,
        n_case            = no_AD_n_cases,
        n_control         = no_AD_n_controls
    )
        
    #==========================
    # Model 4: Raw phenotypes only (EXCLUDE ALL MODULATORS)
    #==========================
    if (ancestry == "European") {
      formula_cyp_no_mod <- status ~ age_at_first_AD + sex_at_birth +
        Education_level + Income_quantitative + num_prior_AD +
        CYP2D6 + CYP2C19 + CYP2B6
    } else {
      formula_cyp_no_mod <- status ~ age_at_first_AD + sex_at_birth +
        Education_level + Income_quantitative + ancestry_pred + num_prior_AD +
        CYP2D6 + CYP2C19 + CYP2B6
    }

    results_list[["CYP_Main_raw_NoMod"]] <- fit_glm_and_tidy(
      glm_formula       = formula_cyp_no_mod,
      model_name        = "CYP_Main_raw_NoMod",
      independent_label = "CYP_raw_ExcludeAllModulators",
      data              = no_any_mod_dat,
      n_total           = no_any_n_totals,
      n_case            = no_any_n_cases,
      n_control         = no_any_n_controls
    )
      
      #== Combine results
      results_list <- results_list[!sapply(results_list, is.null)]
      
      if (length(results_list) > 0) {
        results_df <- dplyr::bind_rows(results_list)
        
        results_format <- results_df %>%
          select(
            Drug, Ancestry, Window, Model,
            N_Total, N_Cases, N_Controls,
            variable, odds_ratio, lower_95, upper_95, p_value
          ) %>%
          mutate(
            variable    = gsub(":", "_", variable),
            variable    = gsub("_adjusted", "_Ph", variable),
            variable    = gsub("Poor/Intermediate", "_P/I", variable),
            variable    = gsub("Normal",      "_N", variable),
            variable    = gsub("Poor",        "_P", variable),
            variable    = gsub("Ultrarapid",  "_U", variable),
            variable    = gsub("Rapid",       "_R", variable),
            variable    = gsub("Intermediate","_I", variable)
          )
        
        combo <- paste(drug, ancestry, window, sep = "_")
        all_results[[combo]] <- results_format
        
        message("  Completed: ", combo)
      }
      
    } # end ancestry
  } # end window
  
  #== Combine all results for this drug
  if (length(all_results) > 0) {
    drug_results_combined <- bind_rows(all_results)
    drug_results_list[[drug]] <- drug_results_combined
    
    message("\n=== Finished drug: ", drug, " ===\n")
  }
  
} # end drug loop


In [ ]:
# Combine all drugs
final_results <- bind_rows(drug_results_list)

fdr_subset <- function(p_all, mask) {
  out <- rep(NA_real_, length(p_all))
  out[mask] <- p.adjust(p_all[mask], method = "BH")
  out
}

# Apply FDR
final_results <- final_results %>%
  group_by(Ancestry, Window, Model) %>%
  mutate(
    p_fdr = if (startsWith(Model[1], "Cov")) {
      fdr_subset(p_value, startsWith(variable, "cyp"))
    } else {
      fdr_subset(p_value, startsWith(variable, "CYP"))
    }
  ) %>%
  ungroup()

final_results <- final_results %>%
  group_by(Ancestry, Window, Model, Drug) %>%
  mutate(
    p_fdr_drug = if (startsWith(Model[1], "Cov")) {
      fdr_subset(p_value, startsWith(variable, "cyp"))
    } else {
      fdr_subset(p_value, startsWith(variable, "CYP"))
    }
  ) %>%
  ungroup()

In [ ]:
# Check FDR is applied correctly
final_results %>%
  group_by(Ancestry, Window, Model) %>%
  summarise(
    n_tests = n(),
    n_sig_nom = sum(p_value < 0.05),
    n_sig_fdr = sum(p_fdr < 0.05, na.rm=T),
    .groups = "drop"
  )

In [ ]:
#==============================================================================
# SEX STRATIFICATION SENSITIVITY ANALYSIS 
#==============================================================================
sex_groups <- c("Male", "Female")
drug_results_sex_strat <- list()

for (drug in drugs) {
  
  print(paste("Processing sex stratification for drug:", drug))
  
  #== Repeat the same data prep as before (first_switch, no_switch, etc.)
    #-- First switch (first incidence of switch)
    first_switch <- outcome %>%
      filter(drug.treatment == drug) %>%
      filter(FinalClassification.treatment == "Switching") %>%
      group_by(person_id) %>%
      filter(Incidence == min(Incidence)) %>%
      filter(FirstDispense.treatment.episode == min(FirstDispense.treatment.episode)) %>%
      filter(ExpectedEndDate.treatment.episode == max(ExpectedEndDate.treatment.episode)) %>%
      filter(ExpectedEndDate.treatment == min(ExpectedEndDate.treatment)) %>%
      ungroup()

    #-- No switch (first incidence of continuation among non-switchers)
    no_switch <- outcome %>%
      filter(drug.treatment == drug) %>%
      filter(!person_id %in% first_switch$person_id) %>%
      filter(FinalClassification.treatment == "Continuation") %>%
      group_by(person_id) %>%
      arrange(Incidence, FirstDispense.treatment) %>%
      slice(1) %>%
      ungroup()
  
  dat <- rbind(first_switch, no_switch)
    
  #==== Find number of other AD dispensed prior index date of drug
  first_switch_other <- dat %>%
    left_join(outcome, by = c("person_id"), suffix = c(".first", ".second")) %>%
    filter(FirstDispense.treatment.episode.second < Index.first) %>%
    filter(drug.treatment.first != drug.treatment.second)
  
  count <- first_switch_other %>%
    group_by(person_id) %>%
    summarise(num_prior_AD = length(unique(drug.treatment.second)))
  
  dat <- dat %>%
    left_join(count, by = "person_id") %>%
    mutate(num_prior_AD = ifelse(is.na(num_prior_AD), 0, num_prior_AD))
  
  input_data <- dat %>%
    left_join(pheno, by = "person_id") %>%
    mutate(
      age_at_first_AD = interval(date_of_birth, DateFirstDispense) %/% years(1)
    )
  
  input_data <- input_data %>%
    mutate(
      status = case_when(
        FinalClassification.treatment == "Switching" ~ 1L,
        FinalClassification.treatment != "Switching" ~ 0L,
        TRUE ~ NA_integer_
      )
    )
    
  #== Collapse CYP2B6 intermediate and Poor into one category
  input_data <- input_data %>%
    mutate(
        CYP2B6 = case_when(
        CYP2B6 %in% c("Poor", "Intermediate") ~ "Poor/Intermediate",
        TRUE ~ CYP2B6)
        )
  
  # Set raw phenotypes as factors
  input_data$CYP2C19 <- factor(input_data$CYP2C19, levels = c("Poor", "Intermediate", "Normal", "Rapid", "Ultrarapid"))
  input_data$CYP2C19 <- stats::relevel(input_data$CYP2C19, ref = "Normal")
  
  input_data$CYP2D6 <- factor(input_data$CYP2D6, levels = c("Poor", "Intermediate", "Normal", "Ultrarapid"))
  input_data$CYP2D6 <- stats::relevel(input_data$CYP2D6, ref = "Normal")
  
  input_data$CYP2B6 <- factor(input_data$CYP2B6, levels = c("Poor/Intermediate", "Normal", "Ultrarapid"))
  input_data$CYP2B6 <- stats::relevel(input_data$CYP2B6, ref = "Normal")
    
  input_data$ancestry_pred <- factor(input_data$ancestry_pred)
  input_data$ancestry_pred <- relevel(input_data$ancestry_pred, ref = "eur")
    
  
  #== Loop through sex groups, windows, and ancestry
  all_results_sex <- list()
  
  for (sex_group in sex_groups) {
    for (window in cyp_windows) {
      for (ancestry in ancestry_groups) {
        
        message(sprintf("\n>>> Drug: %s, Sex: %s, Window: %s, Ancestry: %s",
                        drug, sex_group, window, ancestry))
        
        #== Filter ancestry
        if (ancestry== "European") {
          input_data_pop <- input_data %>%
            filter(ancestry_pred == "eur")
        } else {
          input_data_pop <- input_data
        }
        
        #== Filter by sex group
        input_data_pop <- input_data_pop %>%
          filter(sex_at_birth == sex_group)
        
        # Skip if too few people in sex group
        if (nrow(input_data_pop) < 40) {
          message("  Skipping: Insufficient sample in sex group (N=", nrow(input_data_pop), ")")
          next
        }
        
        #== Map window-specific CYP columns
        c19_adj_col <- paste0("CYP2C19_", window, "_phenoconverted")
        d6_adj_col  <- paste0("CYP2D6_", window, "_phenoconverted")
        b6_adj_col  <- paste0("CYP2B6_", window, "_phenoconverted")
        
        c19_s_inhib_col <- paste0(window, "_cyp2c19_strong_inhibitors")
        c19_m_inhib_col <- paste0(window, "_cyp2c19_moderate_inhibitors")
        c19_w_inhib_col <- paste0(window, "_cyp2c19_weak_inhibitors")
        c19_m_ind_col   <- paste0(window, "_cyp2c19_moderate_inducers")
        c19_w_ind_col   <- paste0(window, "_cyp2c19_weak_inducers")
        
        d6_s_inhib_col <- paste0(window, "_cyp2d6_strong_inhibitors")
        d6_m_inhib_col <- paste0(window, "_cyp2d6_moderate_inhibitors")
        d6_w_inhib_col <- paste0(window, "_cyp2d6_weak_inhibitors")
        
        b6_s_inhib_col <- paste0(window, "_cyp2b6_strong_inhibitors")
        b6_m_inhib_col <- paste0(window, "_cyp2b6_moderate_inhibitors")
        b6_w_inhib_col <- paste0(window, "_cyp2b6_weak_inhibitors")
        b6_m_ind_col   <- paste0(window, "_cyp2b6_moderate_inducers")
        b6_w_ind_col   <- paste0(window, "_cyp2b6_weak_inducers")
          
        # CYP neg controls
        prav_col <- paste0(window, "_pravastatin_neg_cont")
        rosu_col <- paste0(window, "_rosuvastatin_neg_cont")
          
        # AD modulator indicators (already exist in data)
        has_AD_d6_col  <- paste0("has_AD_cyp2d6_modulator_", window)
        has_AD_c19_col <- paste0("has_AD_cyp2c19_modulator_", window)
        
        #== Create window-specific adjusted phenotypes and modulators
        input_data_window <- input_data_pop %>%
          mutate(
            CYP2C19_adjusted = .data[[c19_adj_col]],
            CYP2D6_adjusted  = .data[[d6_adj_col]],
            CYP2B6_adjusted  = .data[[b6_adj_col]],
            
            cyp2c19_strong_inhib = .data[[c19_s_inhib_col]],
            cyp2c19_mod_inhib    = .data[[c19_m_inhib_col]],
            cyp2c19_weak_inhib   = .data[[c19_w_inhib_col]],
            cyp2c19_mod_ind      = .data[[c19_m_ind_col]],
            cyp2c19_weak_ind     = .data[[c19_w_ind_col]],
            
            cyp2d6_strong_inhib = .data[[d6_s_inhib_col]],
            cyp2d6_mod_inhib    = .data[[d6_m_inhib_col]],
            cyp2d6_weak_inhib   = .data[[d6_w_inhib_col]],
            
            cyp2b6_strong_inhib = .data[[b6_s_inhib_col]],
            cyp2b6_mod_inhib    = .data[[b6_m_inhib_col]],
            cyp2b6_weak_inhib   = .data[[b6_w_inhib_col]],
            cyp2b6_mod_ind      = .data[[b6_m_ind_col]],
            cyp2b6_weak_ind     = .data[[b6_w_ind_col]],
              
            # CYP negative controls
            prav_neg_cont        = .data[[prav_col]],
            rosu_neg_cont        = .data[[rosu_col]],
            
            # AD modulator indicators
            has_AD_cyp2d6_modulator  = .data[[has_AD_d6_col]],
            has_AD_cyp2c19_modulator = .data[[has_AD_c19_col]],
            has_any_AD_modulator = as.integer(
            has_AD_cyp2d6_modulator == 1L | has_AD_cyp2c19_modulator == 1L
            ),
              
            ## NEW: Create "any inhibitor" variables
              cyp2d6_any_inhib = as.integer(cyp2d6_strong_inhib == 1 | 
                                           cyp2d6_mod_inhib == 1 | 
                                           cyp2d6_weak_inhib == 1),

              cyp2c19_any_inhib = as.integer(cyp2c19_strong_inhib == 1 | 
                                            cyp2c19_mod_inhib == 1 | 
                                            cyp2c19_weak_inhib == 1),

              cyp2b6_any_inhib = as.integer(cyp2b6_strong_inhib == 1 | 
                                           cyp2b6_mod_inhib == 1 | 
                                           cyp2b6_weak_inhib == 1),

              cyp2c19_any_ind = as.integer(cyp2c19_mod_ind == 1 | 
                                          cyp2c19_weak_ind == 1),

              cyp2b6_any_ind = as.integer(cyp2b6_mod_ind == 1 | 
                                         cyp2b6_weak_ind == 1),

              # AD modulator indicators
              has_AD_cyp2d6_modulator  = .data[[has_AD_d6_col]],
              has_AD_cyp2c19_modulator = .data[[has_AD_c19_col]],
              has_any_AD_modulator = as.integer(
                has_AD_cyp2d6_modulator == 1L | has_AD_cyp2c19_modulator == 1L
              )
        
          )
          
        #-- Combine CYP2B6 strong and moderate inhibitors
        input_data_window <- input_data_window %>%
            mutate(
                cyp2b6_mod_strong_inhib = case_when(
                    (cyp2b6_mod_inhib == 1 | cyp2b6_strong_inhib == 1) ~ 1,
                    TRUE ~ 0),
                cyp2b6_weak_mod_ind = case_when(
                    cyp2b6_weak_ind == 1 | cyp2b6_mod_ind == 1 ~ 1,
                    TRUE ~ 0),
                cyp2c19_weak_mod_ind = case_when(
                    cyp2c19_weak_ind == 1 | cyp2c19_mod_ind == 1 ~ 1,
                    TRUE ~ 0),
                cyp2d6_mod_strong_inhib = case_when(
                    cyp2d6_mod_inhib == 1 | cyp2d6_strong_inhib == 1 ~ 1,
                    TRUE ~ 0)
                ) %>%
        mutate(
            CYP2B6_adjusted = case_when(
                CYP2B6_adjusted %in% c("Poor", "Intermediate") ~ "Poor/Intermediate",
                TRUE ~ CYP2B6_adjusted)
        )
        
        
        #== Set adjusted phenotypes as factors
        input_data_window$CYP2C19_adjusted <- factor(input_data_window$CYP2C19_adjusted, 
                                                      levels = c("Poor", "Intermediate", "Normal", "Rapid", "Ultrarapid"))
        input_data_window$CYP2C19_adjusted <- stats::relevel(input_data_window$CYP2C19_adjusted, ref = "Normal")
        
        input_data_window$CYP2D6_adjusted <- factor(input_data_window$CYP2D6_adjusted,
                                                     levels = c("Poor", "Intermediate", "Normal", "Ultrarapid"))
        input_data_window$CYP2D6_adjusted <- stats::relevel(input_data_window$CYP2D6_adjusted, ref = "Normal")
        
        input_data_window$CYP2B6_adjusted <- factor(input_data_window$CYP2B6_adjusted,
                                                     levels = c("Poor/Intermediate","Normal", "Ultrarapid"))
        input_data_window$CYP2B6_adjusted <- stats::relevel(input_data_window$CYP2B6_adjusted, ref = "Normal")
        
        #== Build complete-case dataset
        common_vars <- c(
          "person_id", "status",
          "age_at_first_AD", "sex_at_birth", "Education_level", "Income_quantitative", 
          "cyp2d6_strong_inhib", "cyp2d6_mod_inhib", "cyp2d6_weak_inhib", "num_prior_AD",
          "cyp2c19_strong_inhib", "cyp2c19_mod_inhib", "cyp2c19_weak_inhib",
          "cyp2c19_weak_mod_ind",
          "cyp2b6_mod_strong_inhib", "cyp2b6_weak_inhib",
          "cyp2b6_weak_mod_ind",
          "CYP2D6", "CYP2C19", "CYP2B6",
          "CYP2D6_adjusted", "CYP2C19_adjusted", "CYP2B6_adjusted",
          "cyp2d6_any_inhib", "cyp2c19_any_inhib", "cyp2b6_any_inhib", "cyp2d6_mod_strong_inhib",
          "cyp2c19_any_ind", "cyp2b6_any_ind", "prav_neg_cont", "rosu_neg_cont",
          "has_AD_cyp2d6_modulator", "has_AD_cyp2c19_modulator", "has_any_AD_modulator", "has_any_AD_modulator_ever"
        )
        
        if (ancestry == "European") {
          base_dat <- input_data_window %>%
            select(all_of(common_vars)) %>%
            filter(complete.cases(.))
        } else {
          base_dat <- input_data_window %>%
            select(all_of(c(common_vars, "ancestry_pred"))) %>%
            filter(complete.cases(.))
        }
        
        if (nrow(base_dat) == 0) {
          message("  Skipping: No data after complete.cases")
          next
        }
          
      #== Create subset excluding AD modulators
      no_AD_mod_dat <- base_dat %>%
        filter(has_any_AD_modulator_ever == 0L)
          
    #== Create subset excluding ALL modulators (any CYP inhibitor/inducer)
        no_any_mod_dat <- base_dat %>%
          filter(cyp2d6_any_inhib == 0 &
                 cyp2c19_any_inhib == 0 &
                 cyp2c19_any_ind == 0 &
                 cyp2b6_any_inhib == 0 &
                 cyp2b6_any_ind == 0)

        # Calculate sample sizes
        no_any_mod_counts <- no_any_mod_dat %>%
          group_by(status) %>%
          summarise(n = n(), .groups = "drop")

        no_any_n_cases    <- if(1 %in% no_any_mod_counts$status) no_any_mod_counts$n[no_any_mod_counts$status == 1] else 0
        no_any_n_controls <- if(0 %in% no_any_mod_counts$status) no_any_mod_counts$n[no_any_mod_counts$status == 0] else 0
        no_any_n_totals   <- no_any_n_cases + no_any_n_controls

        message(sprintf("  No-any-modulator subset: N=%d (Cases=%d, Controls=%d)", 
                        no_any_n_totals, no_any_n_cases, no_any_n_controls))

      # Calculate sample sizes for no-AD-modulator subset
      no_AD_switch_counts <- no_AD_mod_dat %>%
        group_by(status) %>%
        summarise(n = n(), .groups = "drop")
      
      no_AD_n_cases    <- if(1 %in% no_AD_switch_counts$status) no_AD_switch_counts$n[no_AD_switch_counts$status == 1] else 0
      no_AD_n_controls <- if(0 %in% no_AD_switch_counts$status) no_AD_switch_counts$n[no_AD_switch_counts$status == 0] else 0
      no_AD_n_totals   <- no_AD_n_cases + no_AD_n_controls
      
      message(sprintf("  No-AD-modulator subset: N=%d (Cases=%d, Controls=%d)", 
                      no_AD_n_totals, no_AD_n_cases, no_AD_n_controls))
        
        #== Check sample size
        switch_counts <- base_dat %>%
          group_by(status) %>%
          summarise(n = n(), .groups = "drop")
        
        n_cases    <- switch_counts$n[switch_counts$status == 1]
        n_controls <- switch_counts$n[switch_counts$status == 0]
        
        if (length(n_cases) == 0 || length(n_controls) == 0) {
          message("  Skipping: Missing cases or controls")
          next
        }
        
        n_totals <- n_cases + n_controls
        
        # Require minimum sample size
        if (n_cases < 20 || n_controls < 20 || n_totals < 1000) {
          message(sprintf("  Skipping: Insufficient sample (N=%d, Cases=%d, Controls=%d)",
                          n_totals, n_cases, n_controls))
          next
            }
        
        message(sprintf("  N=%d (Cases=%d, Controls=%d)", n_totals, n_cases, n_controls))
        
        results_list <- list()
          
          # Calculate sample sizes for no-AD-modulator subset
        no_AD_switch_counts <- no_AD_mod_dat %>%
          group_by(status) %>%
          summarise(n = n(), .groups = "drop")

        no_AD_n_cases    <- if(1 %in% no_AD_switch_counts$status) no_AD_switch_counts$n[no_AD_switch_counts$status == 1] else 0
        no_AD_n_controls <- if(0 %in% no_AD_switch_counts$status) no_AD_switch_counts$n[no_AD_switch_counts$status == 0] else 0
        no_AD_n_totals   <- no_AD_n_cases + no_AD_n_controls

        message(sprintf("  No-AD-modulator subset: N=%d (Cases=%d, Controls=%d)", 
                        no_AD_n_totals, no_AD_n_cases, no_AD_n_controls))

        # Skip all models if no-AD-modulator subset is too small
        if (no_AD_n_cases < 20 || no_AD_n_controls < 20 || no_AD_n_totals < 1000) {
          message(sprintf("  Skipping all models: no-AD-modulator subset too small (N=%d, Cases=%d, Controls=%d)",
                          no_AD_n_totals, no_AD_n_cases, no_AD_n_controls))
          next
        }
        
        #== Helper function (same as before but adds SexGroup)
        fit_glm_and_tidy <- function(glm_formula, model_name, independent_label,
                                    data = base_dat, 
                                   n_total = n_totals, 
                                   n_case = n_cases, 
                                   n_control = n_controls) {
          dat <- droplevels(data)
          
          if (dplyr::n_distinct(dat$status, na.rm = TRUE) < 2) {
            message("  Skipping model ", model_name, ": outcome has <2 levels")
            return(NULL)
          }
          
          term_labels <- attr(terms(glm_formula), "term.labels")
          
          bad_terms <- term_labels[
            sapply(term_labels, function(v) {
              if (v %in% names(dat)) {
                is.factor(dat[[v]]) && dplyr::n_distinct(dat[[v]], na.rm = TRUE) < 2
              } else {
                FALSE
              }
            })
          ]
          
          if (length(bad_terms) > 0) {
            message("  Dropping factor(s) with <2 levels: ",
                    paste(bad_terms, collapse = ", "))
            glm_formula <- stats::update(glm_formula,
                                         paste(". ~ . -", paste(bad_terms, collapse = " - ")))
          }
            
        # ---- EXPLICIT EDUCATION LEVEL SPARSE CHECK ----
        if ("Education_level" %in% names(dat)) {
          dat$Education_level <- factor(dat$Education_level)

          tbl_ed <- table(dat$Education_level, dat$status, useNA = "no")

          if ("0" %in% colnames(tbl_ed) && "1" %in% colnames(tbl_ed)) {
            sparse_ed <- rownames(tbl_ed)[
              tbl_ed[, "1"] < 2 | tbl_ed[, "0"] < 2
            ]

            if (length(sparse_ed) > 0) {
              # Find a non-sparse level to use as reference
              non_sparse <- setdiff(levels(dat$Education_level), sparse_ed)
              if (length(non_sparse) > 0) {
                dat$Education_level <- relevel(dat$Education_level, ref = non_sparse[1])
              }
              message("  Removing sparse Education_level categories: ",
                      paste(sparse_ed, collapse = ", "))
              dat <- dat[!as.character(dat$Education_level) %in% sparse_ed, ]
              dat$Education_level <- droplevels(dat$Education_level)
            }
          }
        }

        # ---- NEW: DROP SPARSE BINARY MODULATOR TERMS ----
          glm_formula <- drop_sparse_binary_terms(glm_formula, data = dat, min_cell = 2)

          # Fit model
          glm_model <- tryCatch({
            stats::glm(
              glm_formula,
              data   = dat,
              family = stats::binomial(link = "logit")
            )
          }, error = function(e) {
            message("  Error fitting model ", model_name, ": ", e$message)
            return(NULL)
          })
          
          if (is.null(glm_model)) return(NULL)
          
          sm <- summary(glm_model)$coefficients
          
          coef_vec <- sm[, "Estimate"]
          se_vec   <- sm[, "Std. Error"]
          z_vec    <- sm[, "z value"]
          p_vec    <- sm[, "Pr(>|z|)"]
          
          lower_95 <- coef_vec - 1.96 * se_vec
          upper_95 <- coef_vec + 1.96 * se_vec
          
          glm_results <- data.frame(
            variable   = rownames(sm),
            coef       = coef_vec,
            odds_ratio = exp(coef_vec),
            se         = se_vec,
            z          = z_vec,
            p_value    = p_vec,
            lower_95   = exp(lower_95),
            upper_95   = exp(upper_95),
            stringsAsFactors = FALSE
          )
          rownames(glm_results) <- NULL
          
          glm_results$Model      <- model_name
          glm_results$Ancestry <- ancestry
          glm_results$Window     <- window
          glm_results$Drug       <- drug
          glm_results$SexGroup   <- sex_group
          glm_results$N_Total    <- n_total
          glm_results$N_Cases    <- n_case
          glm_results$N_Controls <- n_control
          
          glm_results
        }
        
        #==========================
        # Model 1: Covariate-only (NOTE: sex_at_birth removed from formula since we're stratified by sex)
        #==========================
        if (ancestry == "European") {
          formula_covonly <- status ~ age_at_first_AD +
            Education_level + Income_quantitative + num_prior_AD +
            cyp2d6_mod_strong_inhib + cyp2d6_weak_inhib +
            cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
            cyp2c19_weak_mod_ind + 
            cyp2b6_mod_strong_inhib + cyp2b6_weak_inhib + rosu_neg_cont + prav_neg_cont +
            cyp2b6_weak_mod_ind
        } else {
          formula_covonly <- status ~ age_at_first_AD +
            Education_level + Income_quantitative + ancestry_pred + num_prior_AD +
            cyp2d6_mod_strong_inhib + cyp2d6_weak_inhib +
            cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
            cyp2c19_weak_mod_ind + 
            cyp2b6_mod_strong_inhib + cyp2b6_weak_inhib + rosu_neg_cont + prav_neg_cont +
            cyp2b6_weak_mod_ind
        }
        
        results_list[["CovAll_NoCYP"]] <- fit_glm_and_tidy(
          glm_formula       = formula_covonly,
          model_name        = "CovAll_NoCYP",
          independent_label = "AllCovariates_NoCYP",
          data              = no_AD_mod_dat,
          n_total           = no_AD_n_totals,
          n_case            = no_AD_n_cases,
          n_control         = no_AD_n_controls
        )
          
      #==========================
      # Model 2d: Raw phenotypes (FULL SAMPLE - genotype only)
      #==========================
      if (ancestry == "European") {
        formula_cyp_main_raw <- status ~ age_at_first_AD + 
          Education_level + Income_quantitative + num_prior_AD +
          CYP2D6 + CYP2C19 + CYP2B6
      } else {
        formula_cyp_main_raw <- status ~ age_at_first_AD +
          Education_level + Income_quantitative + ancestry_pred + num_prior_AD +
          CYP2D6 + CYP2C19 + CYP2B6
      }
      
      results_list[["CYP_Main_raw"]] <- fit_glm_and_tidy(
        glm_formula       = formula_cyp_main_raw,
        model_name        = "CYP_Main_raw",
        independent_label = "CYP2C19_CYP2D6_CYP2B6_raw",
        data              = base_dat,
        n_total           = n_totals,
        n_case            = n_cases,
        n_control         = n_controls
      )
        
        
        #==========================
        # Model 2a: Raw phenotypes
        #==========================
        if (ancestry== "European") {
          formula_cyp_main_raw <- status ~ age_at_first_AD +
            Education_level + Income_quantitative + num_prior_AD +
            cyp2d6_mod_strong_inhib + cyp2d6_weak_inhib +
            cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
            cyp2c19_weak_mod_ind + 
            cyp2b6_mod_strong_inhib + cyp2b6_weak_inhib +
            cyp2b6_weak_mod_ind + 
            CYP2D6 + CYP2C19 + CYP2B6
        } else {
          formula_cyp_main_raw <- status ~ age_at_first_AD +
            Education_level + Income_quantitative + ancestry_pred + num_prior_AD +
            cyp2d6_mod_strong_inhib + cyp2d6_weak_inhib +
            cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
            cyp2c19_weak_mod_ind + 
            cyp2b6_mod_strong_inhib + cyp2b6_weak_inhib +
            cyp2b6_weak_mod_ind +
            CYP2D6 + CYP2C19 + CYP2B6
        }
        
        results_list[["CYP_Main_raw_plus_Mod"]] <- fit_glm_and_tidy(
          glm_formula       = formula_cyp_main_raw,
          model_name        = "CYP_Main_raw_plus_Mod",
          independent_label = "CYP2C19_CYP2D6_CYP2B6_raw_plus_Mod",
          data              = no_AD_mod_dat,
          n_total           = no_AD_n_totals,
          n_case            = no_AD_n_cases,
          n_control         = no_AD_n_controls
        )
        
        #==========================
        # Model 2b: Adjusted phenotypes (no modulators)
        #==========================
        if (ancestry == "European") {
          formula_cyp_main_adj <- status ~ age_at_first_AD +
            Education_level + Income_quantitative + num_prior_AD +
            CYP2D6_adjusted + CYP2C19_adjusted + CYP2B6_adjusted
        } else {
          formula_cyp_main_adj <- status ~ age_at_first_AD +
            Education_level + Income_quantitative + ancestry_pred + num_prior_AD +
            CYP2D6_adjusted + CYP2C19_adjusted + CYP2B6_adjusted
        }
        
        results_list[["CYP_Main_adj"]] <- fit_glm_and_tidy(
          glm_formula       = formula_cyp_main_adj,
          model_name        = "CYP_Main_adj",
          independent_label = "CYP2C19_CYP2D6_CYP2B6_adj",
          data              = no_AD_mod_dat,
          n_total           = no_AD_n_totals,
          n_case            = no_AD_n_cases,
          n_control         = no_AD_n_controls
        )
        
        #==========================
        # Model 2c: Adjusted phenotypes + modulators
        #==========================
        if (ancestry == "European") {
          formula_cyp_main_adj_mod <- status ~ age_at_first_AD +
            Education_level + Income_quantitative + num_prior_AD +
            cyp2d6_mod_strong_inhib + cyp2d6_weak_inhib +
            cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
            cyp2c19_weak_mod_ind + 
            cyp2b6_mod_strong_inhib + cyp2b6_weak_inhib +
            cyp2b6_weak_mod_ind + 
            CYP2D6_adjusted + CYP2C19_adjusted + CYP2B6_adjusted
        } else {
          formula_cyp_main_adj_mod <- status ~ age_at_first_AD +
            Education_level + Income_quantitative + ancestry_pred + num_prior_AD +
            cyp2d6_mod_strong_inhib + cyp2d6_weak_inhib +
            cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
            cyp2c19_weak_mod_ind + 
            cyp2b6_mod_strong_inhib + cyp2b6_weak_inhib +
            cyp2b6_weak_mod_ind +
            CYP2D6_adjusted + CYP2C19_adjusted + CYP2B6_adjusted
        }
        
        results_list[["CYP_Main_adj_plus_Mod"]] <- fit_glm_and_tidy(
          glm_formula       = formula_cyp_main_adj_mod,
          model_name        = "CYP_Main_adj_plus_Mod",
          independent_label = "CYP2C19_CYP2D6_CYP2B6_adj_mod",
          data              = no_AD_mod_dat,
          n_total           = no_AD_n_totals,
          n_case            = no_AD_n_cases,
          n_control         = no_AD_n_controls
        )
          
        #==========================
        # Model 4: Raw phenotypes only (EXCLUDE ALL MODULATORS)
        #==========================
        if (ancestry == "European") {
          formula_cyp_no_mod <- status ~ age_at_first_AD +
            Education_level + Income_quantitative + num_prior_AD +
            CYP2D6 + CYP2C19 + CYP2B6
        } else {
          formula_cyp_no_mod <- status ~ age_at_first_AD + 
            Education_level + Income_quantitative + ancestry_pred + num_prior_AD +
            CYP2D6 + CYP2C19 + CYP2B6
        }

        results_list[["CYP_Main_raw_NoMod"]] <- fit_glm_and_tidy(
          glm_formula       = formula_cyp_no_mod,
          model_name        = "CYP_Main_raw_NoMod",
          independent_label = "CYP_raw_ExcludeAllModulators",
          data              = no_any_mod_dat,
          n_total           = no_any_n_totals,
          n_case            = no_any_n_cases,
          n_control         = no_any_n_controls
        )
        
        #== Combine results
        results_list <- results_list[!sapply(results_list, is.null)]
        
        if (length(results_list) > 0) {
          results_df <- dplyr::bind_rows(results_list)
          
          results_format <- results_df %>%
            select(
              Drug, Ancestry, Window, Model, SexGroup,
              N_Total, N_Cases, N_Controls,
              variable, odds_ratio, lower_95, upper_95, p_value
            ) %>%
            mutate(
              variable    = gsub(":", "_", variable),
              variable    = gsub("_adjusted", "_Ph", variable),
              variable    = gsub("Poor/Intermediate", "_I/P", variable),
              variable    = gsub("Normal",      "_N", variable),
              variable    = gsub("Poor",        "_P", variable),
              variable    = gsub("Ultrarapid",  "_U", variable),
              variable    = gsub("Rapid",       "_R", variable),
              variable    = gsub("Intermediate","_I", variable)
            )
          
          combo <- paste(drug, ancestry, window, sex_group, sep = "_")
          all_results_sex[[combo]] <- results_format
          
          message("  Completed: ", combo)
        }
        
      } # end ancestry
    } # end window
  } # end sex_group
  
  #== Combine all results for this drug
  if (length(all_results_sex) > 0) {
    drug_results_combined <- bind_rows(all_results_sex)
    drug_results_sex_strat[[drug]] <- drug_results_combined
    
    message("\n=== Finished sex stratification for drug: ", drug, " ===\n")
  }
  
} # end drug loop for sex stratification


# Combine all drugs
final_results_with_sex <- bind_rows(drug_results_sex_strat)


In [ ]:
# Apply FDR
final_results_with_sex <- final_results_with_sex %>%
  group_by(Ancestry, Window, Model, SexGroup) %>%
  mutate(
    p_fdr = if (startsWith(Model[1], "Cov")) {
      fdr_subset(p_value, startsWith(variable, "cyp"))
    } else {
      fdr_subset(p_value, startsWith(variable, "CYP"))
    }
  ) %>%
  ungroup()

final_results_with_sex <- final_results_with_sex %>%
  group_by(Ancestry, Window, Model, SexGroup, Drug) %>%
  mutate(
    p_fdr_drug = if (startsWith(Model[1], "Cov")) {
      fdr_subset(p_value, startsWith(variable, "cyp"))
    } else {
      fdr_subset(p_value, startsWith(variable, "CYP"))
    }
  ) %>%
  ungroup()

# Clean up and add significance flags
final_results_with_sex <- final_results_with_sex %>%
  select(
    SexGroup, Drug, Ancestry, Window, Model,
    N_Total, N_Cases, N_Controls,
    variable, odds_ratio, lower_95, upper_95,
    p_value, p_fdr, p_fdr_drug
  )

In [ ]:
# Add SexGroup = "All" to your main results for easy comparison
final_results <- final_results %>%
  mutate(SexGroup = "All")

# Combine for comparison
combined_all_and_strat <- bind_rows(
  final_results,
  final_results_with_sex
)
combined_all_and_strat <- combined_all_and_strat %>%
  mutate(
    variable = gsub("Ph_", "", variable)
  )

#-- Replace "concom_only" with Post-index only
combined_all_and_strat <- combined_all_and_strat %>%
    mutate(Window = case_when(
        Window == "concom_only" ~ "post_index_only",
        TRUE ~ Window))


In [ ]:
#-- Concatenate, reformat and save results
all_df <- combined_all_and_strat %>%
    select(Ancestry, Window, SexGroup, Model, Drug, variable, odds_ratio, lower_95, upper_95, p_value, p_fdr, p_fdr_drug, N_Total, N_Cases, N_Controls)

#-- rename Full Ancestry to All Ancestry
all_df <- all_df %>%
    mutate(Ancestry = case_when(
        Ancestry == "Full" ~ "All", 
        TRUE ~ Ancestry))

# Main results (all ancestries)
table3 <- all_df %>% 
  filter((Ancestry != "European" & Window == "any_followup" & SexGroup == "All" & !Model %in% c("CovAll_NoCYP", "Any_Inhib")))

# Sensitivity: European
table4 <- all_df %>% 
  filter(Ancestry == "European" & Window == "any_followup" & SexGroup == "All" & Model == "CYP_Main_adj")

# Sensitivity: Temporal windows 
table5 <- all_df %>% 
  filter(Ancestry != "European" & Window != "any_followup" & SexGroup == "All" & Model == "CYP_Main_adj")

# Sensitivity: Sex-stratified
table6 <- all_df %>% 
  filter(Ancestry != "European" & Window == "any_followup" & SexGroup != "All" & Model == "CYP_Main_adj")

# Covariate models (NO CYP) - remove the "any inhib model"
table7 <- all_df %>%
    filter(Model %in% c("CovAll_NoCYP") & Ancestry != "European" & Window == "any_followup" & SexGroup == "All")

In [ ]:
#system(paste0("gsutil cp ", Sys.getenv("WORKSPACE_BUCKET"), "/results/All_Results_CYP.xlsx /home/jupyter/workspaces/cypphenoconversion/data/All_Results_CYP.xlsx"))
wb <- loadWorkbook("/home/jupyter/workspaces/cypphenoconversion/data/All_Results_CYP.xlsx")

removeWorksheet(wb, "DrugSpecific_European")
addWorksheet(wb, "DrugSpecific_European")
writeData(wb, "DrugSpecific_European", table4)

removeWorksheet(wb, "DrugSpecific_All_Ancestries")
addWorksheet(wb, "DrugSpecific_All_Ancestries")
writeData(wb, "DrugSpecific_All_Ancestries", table3)

removeWorksheet(wb, "DrugSpecific_TemporalWindows")
addWorksheet(wb, "DrugSpecific_TemporalWindows")
writeData(wb, "DrugSpecific_TemporalWindows", table5)

removeWorksheet(wb, "DrugSpecific_SexStratified")
addWorksheet(wb, "DrugSpecific_SexStratified")
writeData(wb, "DrugSpecific_SexStratified", table6)

removeWorksheet(wb, "DrugSpecific_Covariates")
addWorksheet(wb, "DrugSpecific_Covariates")
writeData(wb, "DrugSpecific_Covariates", table7)

saveWorkbook(wb, "/home/jupyter/workspaces/cypphenoconversion/data/All_Results_CYP.xlsx", overwrite=TRUE)


# Copy to workspace bucket
system(paste(
  "gsutil -u", Sys.getenv("GOOGLE_PROJECT"), "cp /home/jupyter/workspaces/cypphenoconversion/data/All_Results_CYP.xlsx",
  file.path(Sys.getenv("WORKSPACE_BUCKET"), "results/All_Results_CYP.xlsx")
))

In [ ]:
#-- Read in data
library(dplyr)
library(stringr)
library(forcats)
library(ggplot2)
library(readxl)

#-- All ancestries (phenoconverted + mods)
all_df <- read_excel("/home/jupyter/workspaces/cypphenoconversion/data/All_Results_CYP.xlsx", sheet = "DrugSpecific_All_Ancestries")
unique(all_df$variable)

plot_data <- all_df %>%
  filter(Model == "CYP_Main_adj_plus_Mod") %>%
  filter(str_detect(variable, "CYP")) %>%
  filter(Ancestry == "All") %>%
  mutate(Drug = gsub("Serotonin_Modulator", "SM", Drug)) %>%
  filter(Window == "any_followup") %>%
  filter(SexGroup == "All")

n_drugs <- length(unique(plot_data$Drug))

plot_data <- plot_data %>%
    mutate(Sig_FDR_drug = 
           case_when(p_fdr_drug < 0.05 ~ "Sig", 
                     p_fdr_drug >= 0.05 ~ "NotSig",
                    is.na(p_fdr_drug) ~ "NotSig",
                    TRUE ~ NA_character_),
          Sig_FDR = 
          case_when(p_fdr < 0.05 ~ "Sig", 
                     p_fdr >= 0.05 ~ "NotSig",
                    is.na(p_fdr) ~ "NotSig",
                    TRUE ~ NA_character_),
          Sig_Nom = 
          case_when(p_value < 0.05 & p_fdr_drug >= 0.05 ~ TRUE, 
                     p_value >= 0.05 ~ FALSE,
                    is.na(p_value) ~ FALSE,
                    TRUE ~ NA))

plot_data$SigColour <- ifelse(
  plot_data$Sig_FDR_drug == "Sig", 
  "Sig", 
  "NotSig"
)

plot_data$variable = factor(plot_data$variable, levels = c("CYP2D6_P", "CYP2D6_I", "CYP2D6_U",
                                                          "CYP2C19_P", "CYP2C19_I", "CYP2C19_R", "CYP2C19_U",
                                                          "CYP2B6_P/I", "CYP2B6_U"))
plot_data <- plot_data %>%
    mutate(Gene = case_when(
    str_detect(variable, "CYP2D6") ~ "CYP2D6",
    str_detect(variable, "CYP2B6") ~ "CYP2B6",
    TRUE ~ "CYP2C19"))


fill_colors <- c(
  "CYP2D6" = "#c4253d",
  "CYP2C19" = "#FD8D3C",
  "CYP2B6" = "#e9b43b"
)
plot_data <- plot_data %>%
    mutate(Drug = paste0(Drug, "\nN=", N_Total, " (", round(N_Cases/N_Total*100, 0), "%)"))

In [ ]:
plot_data <- plot_data %>%
  mutate(
    SigFill = if_else(Sig_FDR == "Sig", Gene, "NotSig")
  )

fill_colors <- c(
  "CYP2D6"  = "#c4253d",
  "CYP2C19" = "#FD8D3C",
  "CYP2B6"  = "#e9b43b"
)

plot_data$variable <- factor(plot_data$variable,
   levels = c(
      "CYP2B6_P/I",  "CYP2B6_N", "CYP2B6_U",
      "CYP2C19_P","CYP2C19_I","CYP2C19_R","CYP2C19_N","CYP2C19_U",
      "CYP2D6_P","CYP2D6_I","CYP2D6_N","CYP2D6_U"
))

pos <- position_dodge(width = 0.5)

p1 <- ggplot(plot_data, aes(x = odds_ratio, y = variable)) +
  geom_vline(xintercept = 1, linetype = "dashed") +
  geom_errorbarh(
    aes(xmin = lower_95, xmax = upper_95, colour = Gene),
    position = pos
  ) +
  geom_point(
    aes(colour = Gene, fill = SigFill),
    position = pos,
    shape  = 21,
    size   = 2.5,
    stroke = 0.7
  ) +

  labs(
    x = "Odds ratio (95% CI, log-scale)",
    y = "Phenoconversion-adjusted metabolizer status",
      title = NULL
  ) +
  theme_minimal() +
  theme(
      strip.background = element_rect(fill="grey92", colour=NA),
    strip.text.y = element_text(angle = 0),
    strip.text.x = element_text(angle = 0),
    axis.text    = element_text(color = "black"),
    strip.text   = element_text(face = "bold"),
    axis.title = element_text(face = "bold"),
    legend.title = element_text(face = "bold"),
    legend.position = "bottom",
    plot.title = element_text(face = "bold")
  ) +
  scale_color_manual(values = fill_colors, name = "Gene") +
  scale_fill_manual(
    values = c(fill_colors, NotSig = "white"),
    guide  = "none"
  ) +
  facet_wrap(Drug ~ ., ncol = 3)+
  geom_text(
    data = subset(plot_data, Sig_Nom),
    aes(
      label = "*",
      x = upper_95 + 1
    ),
    position = pos,
    size = 4,
    colour = "black"
  ) +
  scale_x_log10()

p1


In [ ]:
# Save as PNG
ggsave(
  filename = "/home/jupyter/workspaces/cypphenoconversion/data/Figures/Supp_DrugSpecific_All_Pheno_wMod.png",
  plot     = p1,
  width    = 8,
  height   = 8,
  dpi      = 600
)

In [ ]:
#-- Main plot (All ancestries and phenoconverted effects without modulators)
plot_data <- all_df %>%
  filter(Model == "CYP_Main_adj") %>%
  filter(str_detect(variable, "CYP")) %>%
  filter(Ancestry == "All") %>%
  mutate(Drug = gsub("Serotonin_Modulator", "SM", Drug)) %>%
  filter(Window == "any_followup") %>%
  filter(SexGroup == "All")

n_drugs <- length(unique(plot_data$Drug))

plot_data<- plot_data %>%
        mutate(Sig_FDR_drug = 
           case_when(p_fdr_drug < 0.05 ~ "Sig", 
                     p_fdr_drug >= 0.05 ~ "NotSig",
                    is.na(p_fdr_drug) ~ "NotSig",
                    TRUE ~ NA_character_),
          Sig_FDR = 
          case_when(p_fdr < 0.05 ~ "Sig", 
                     p_fdr >= 0.05 ~ "NotSig",
                    is.na(p_fdr) ~ "NotSig",
                    TRUE ~ NA_character_),
          Sig_Nom = 
          case_when(p_value < 0.05 & p_fdr_drug >= 0.05 ~ TRUE, 
                     p_value >= 0.05 ~ FALSE,
                    is.na(p_value) ~ FALSE,
                    TRUE ~ NA))

plot_data$SigColour <- ifelse(
  plot_data$Sig_FDR_drug == "Sig", 
  "Sig", 
  "NotSig"
)

plot_data$variable = factor(plot_data$variable, levels = c("CYP2D6_P", "CYP2D6_I", "CYP2D6_U",
                                                          "CYP2C19_P", "CYP2C19_I", "CYP2C19_R", "CYP2C19_U",
                                                          "CYP2B6_P/I", "CYP2B6_U"))
plot_data <- plot_data %>%
    mutate(Gene = case_when(
    str_detect(variable, "CYP2D6") ~ "CYP2D6",
    str_detect(variable, "CYP2B6") ~ "CYP2B6",
    TRUE ~ "CYP2C19"))


fill_colors <- c(
  "CYP2D6" = "#c4253d",
  "CYP2C19" = "#FD8D3C",
  "CYP2B6" = "#e9b43b"
)
plot_data <- plot_data %>%
    mutate(Drug = paste0(Drug, "\nN=", N_Total, " (", round(N_Cases/N_Total*100, 0), "%)"))

In [ ]:
plot_data <- plot_data %>%
  mutate(
    SigFill = if_else(Sig_FDR_drug == "Sig", Gene, "NotSig")
  )

fill_colors <- c(
  "CYP2D6"  = "#c4253d",
  "CYP2C19" = "#FD8D3C",
  "CYP2B6"  = "#e9b43b"
)

plot_data$variable <- factor(plot_data$variable,
   levels = c(
      "CYP2B6_P/I", "CYP2B6_N", "CYP2B6_U",
      "CYP2C19_P","CYP2C19_I","CYP2C19_R","CYP2C19_N","CYP2C19_U",
      "CYP2D6_P","CYP2D6_I","CYP2D6_N","CYP2D6_U"
))

pos <- position_dodge(width = 0.5)
plot_data$Drug = factor(plot_data$Drug, levels = c("SSRI:sertraline\nN=6626 (31%)",
                                                     "SNRI:venlafaxine\nN=3078 (34%)",
                                                     "TCA:amitriptyline\nN=2680 (27%)",
                                                     "SSRI:escitalopram\nN=4432 (26%)",
                                                     "SNRI:duloxetine\nN=4101 (26%)",
                                                     "NDRI:bupropion\nN=4842 (29%)",
                                                     "NaSSA:mirtazapine\nN=1194 (39%)",
                                                     "SSRI:citalopram\nN=3877 (31%)",
                                                     "SSRI:fluoxetine\nN=3663 (29%)", 
                                                     "SSRI:paroxetine\nN=1462 (27%)",
                                                     "SARI:trazodone\nN=3813 (34%)",
                                                     "TCA:nortriptyline\nN=1495 (27%)"))

p1 <- ggplot(plot_data, aes(x = odds_ratio, y = variable)) +
  geom_vline(xintercept = 1, linetype = "dashed") +
  geom_errorbarh(
    aes(xmin = lower_95, xmax = upper_95, colour = Gene),
    position = pos
  ) +
  geom_point(
    aes(colour = Gene, fill = SigFill),
    position = pos,
    shape  = 21,
    size   = 2.5,
    stroke = 0.7
  ) +

  labs(
    x = "Odds ratio (95% CI, log-scale)",
    y = "Phenoconversion-adjusted metabolizer status",
      title = NULL
  ) +
  theme_minimal() +
  theme(
    strip.background = element_rect(fill="grey92", colour=NA),
    strip.text.y = element_text(angle = 0),
    strip.text.x = element_text(angle = 0),
    axis.text    = element_text(color = "black"),
    strip.text   = element_text(face = "bold"),
    axis.title = element_text(face = "bold"),
    legend.title = element_text(face = "bold"),
    legend.position = "bottom",
    plot.title = element_text(face = "bold")
  ) +
  scale_color_manual(values = fill_colors, name = "Gene") +
  scale_fill_manual(
    values = c(fill_colors, NotSig = "white"),
    guide  = "none"
  ) +
  facet_wrap(Drug ~ ., ncol = 3) +
  scale_x_log10()


p1


In [ ]:
# Save as PNG
ggsave(
  filename = "/home/jupyter/workspaces/cypphenoconversion/data/Figures/Main_SpecificDrug_All_Pheno.png",
  plot     = p1,
  width    = 8,
  height   = 8,
  dpi      = 600
)

In [ ]:
#-- All ancestries and raw genotype effects
all_df <- read_excel("/home/jupyter/workspaces/cypphenoconversion/data/All_Results_CYP.xlsx", sheet = "DrugSpecific_All_Ancestries")

plot_data <- all_df %>%
  filter(Model == "CYP_Main_raw") %>%
  filter(str_detect(variable, "CYP")) %>%
  filter(Ancestry == "All") %>%
  mutate(Drug = gsub("Serotonin_Modulator", "SM", Drug)) %>%
  filter(Window == "any_followup")  %>%
  filter(SexGroup == "All")

n_drugs <- length(unique(plot_data$Drug))

plot_data<- plot_data %>%
        mutate(Sig_FDR_drug = 
           case_when(p_fdr_drug < 0.05 ~ "Sig", 
                     p_fdr_drug >= 0.05 ~ "NotSig",
                    is.na(p_fdr_drug) ~ "NotSig",
                    TRUE ~ NA_character_),
          Sig_FDR = 
          case_when(p_fdr < 0.05 ~ "Sig", 
                     p_fdr >= 0.05 ~ "NotSig",
                    is.na(p_fdr) ~ "NotSig",
                    TRUE ~ NA_character_),
          Sig_Nom = 
          case_when(p_value < 0.05 & p_fdr_drug >= 0.05 ~ TRUE, 
                     p_value >= 0.05 ~ FALSE,
                    is.na(p_value) ~ FALSE,
                    TRUE ~ NA))

plot_data$SigColour <- ifelse(
  plot_data$Sig_FDR_drug == "Sig", 
  "Sig", 
  "NotSig"
)

plot_data$variable <- factor(plot_data$variable,
   levels = c(
      "CYP2B6_P/I", "CYP2B6_N", "CYP2B6_U",
      "CYP2C19_P","CYP2C19_I","CYP2C19_R","CYP2C19_N","CYP2C19_U",
      "CYP2D6_P","CYP2D6_I","CYP2D6_N","CYP2D6_U"
))

plot_data <- plot_data %>%
    mutate(Gene = case_when(
    str_detect(variable, "CYP2D6") ~ "CYP2D6",
    str_detect(variable, "CYP2B6") ~ "CYP2B6",
    TRUE ~ "CYP2C19"))


fill_colors <- c(
  "CYP2D6" = "#c4253d",
  "CYP2C19" = "#FD8D3C",
  "CYP2B6" = "#e9b43b"
)
plot_data <- plot_data %>%
    mutate(Drug = paste0(Drug, "\nN=", N_Total, " (", round(N_Cases/N_Total*100, 0), "%)"))

In [ ]:
plot_data <- plot_data %>%
  mutate(
    SigFill = if_else(Sig_FDR_drug == "Sig", Gene, "NotSig")
  )

fill_colors <- c(
  "CYP2D6"  = "#c4253d",
  "CYP2C19" = "#FD8D3C",
  "CYP2B6"  = "#e9b43b"
)

pos <- position_dodge(width = 0.5)

p1 <- ggplot(plot_data, aes(x = odds_ratio, y = variable)) +
  geom_vline(xintercept = 1, linetype = "dashed") +
  geom_errorbarh(
    aes(xmin = lower_95, xmax = upper_95, colour = Gene),
    position = pos
  ) +
  geom_point(
    aes(colour = Gene, fill = SigFill),
    position = pos,
    shape  = 21,
    size   = 2.5,
    stroke = 0.7
  ) +

  labs(
    x = "Odds ratio (95% CI, log-scale)",
    y = "Genetic metabolizer status"
  ) +
  theme_minimal() +
  theme(
    strip.background = element_rect(fill="grey92", colour=NA),
    strip.text.y = element_text(angle = 0),
    strip.text.x = element_text(angle = 0),
    axis.text    = element_text(color = "black"),
    strip.text   = element_text(face = "bold"),
    axis.title = element_text(face = "bold"),
    legend.title = element_text(face = "bold"),
    legend.position = "bottom"
  ) +
  scale_color_manual(values = fill_colors, name = "Gene") +
  scale_fill_manual(
    values = c(fill_colors, NotSig = "white"),
    guide  = "none"
  ) +

  facet_wrap(Drug ~ ., ncol = 3) +
  scale_x_log10()

p1


In [ ]:
# Save as PNG
ggsave(
  filename = "/home/jupyter/workspaces/cypphenoconversion/data/Figures/Supp_DrugSpecific_All_Genetic.png",
  plot     = p1,
  width    = 8,
  height   = 8,
  dpi      = 600
)


In [ ]:
#-- All ancestries and raw genotype effects plus Modulators
plot_data <- all_df %>%
  filter(Model == "CYP_Main_raw_plus_Mod") %>%
  filter(str_detect(variable, "CYP")) %>%
  filter(Ancestry == "All") %>%
  mutate(Drug = gsub("Serotonin_Modulator", "SM", Drug)) %>%
  filter(Window == "any_followup")  %>%
  filter(SexGroup == "All")

n_drugs <- length(unique(plot_data$Drug))

plot_data<- plot_data %>%
        mutate(Sig_FDR_drug = 
           case_when(p_fdr_drug < 0.05 ~ "Sig", 
                     p_fdr_drug >= 0.05 ~ "NotSig",
                    is.na(p_fdr_drug) ~ "NotSig",
                    TRUE ~ NA_character_),
          Sig_FDR = 
          case_when(p_fdr < 0.05 ~ "Sig", 
                     p_fdr >= 0.05 ~ "NotSig",
                    is.na(p_fdr) ~ "NotSig",
                    TRUE ~ NA_character_),
          Sig_Nom = 
          case_when(p_value < 0.05 & p_fdr_drug >= 0.05 ~ TRUE, 
                     p_value >= 0.05 ~ FALSE,
                    is.na(p_value) ~ FALSE,
                    TRUE ~ NA))


plot_data$SigColour <- ifelse(
  plot_data$Sig_FDR_drug == "Sig", 
  "Sig", 
  "NotSig"
)

plot_data$variable <- factor(plot_data$variable,
   levels = c(
      "CYP2B6_P/I", "CYP2B6_N", "CYP2B6_U",
      "CYP2C19_P","CYP2C19_I","CYP2C19_R","CYP2C19_N","CYP2C19_U",
      "CYP2D6_P","CYP2D6_I","CYP2D6_N","CYP2D6_U"
))


plot_data <- plot_data %>%
    mutate(Gene = case_when(
    str_detect(variable, "CYP2D6") ~ "CYP2D6",
    str_detect(variable, "CYP2B6") ~ "CYP2B6",
    TRUE ~ "CYP2C19"))


fill_colors <- c(
  "CYP2D6" = "#c4253d",
  "CYP2C19" = "#FD8D3C",
  "CYP2B6" = "#e9b43b"
)
plot_data <- plot_data %>%
    mutate(Drug = paste0(Drug, "\nN=", N_Total, " (", round(N_Cases/N_Total*100, 0), "%)"))

In [ ]:

plot_data <- plot_data %>%
  mutate(
    SigFill = if_else(Sig_FDR_drug == "Sig", Gene, "NotSig")
  )

fill_colors <- c(
  "CYP2D6"  = "#c4253d",
  "CYP2C19" = "#FD8D3C",
  "CYP2B6"  = "#e9b43b"
)

pos <- position_dodge(width = 0.5)

p1 <- ggplot(plot_data, aes(x = odds_ratio, y = variable)) +
  geom_vline(xintercept = 1, linetype = "dashed") +
  geom_errorbarh(
    aes(xmin = lower_95, xmax = upper_95, colour = Gene),
    position = pos
  ) +
  geom_point(
    aes(colour = Gene, fill = SigFill),
    position = pos,
    shape  = 21,
    size   = 2.5,
    stroke = 0.7
  ) +

  labs(
    x = "Odds ratio (95% CI, log-scale)",
    y = "Genetic metabolizer status"
  ) +
  theme_minimal() +
  theme(
    strip.background = element_rect(fill="grey92", colour=NA),
    strip.text.y = element_text(angle = 0),
    strip.text.x = element_text(angle = 0),
    axis.text    = element_text(color = "black"),
    strip.text   = element_text(face = "bold"),
    axis.title = element_text(face = "bold"),
    legend.title = element_text(face = "bold"),
    legend.position = "bottom"
  ) +
  scale_color_manual(values = fill_colors, name = "Gene") +
  scale_fill_manual(
    values = c(fill_colors, NotSig = "white"),
    guide  = "none"
  ) +

  facet_wrap(Drug ~ ., ncol = 3) +
  geom_text(
    data = subset(plot_data, Sig_Nom),
    aes(
      label = "*",
      x = upper_95 + 1
    ),
    position = pos,
    size = 4,
    colour = "black"
  ) +
  scale_x_log10()

p1


In [ ]:
# Save as PNG
ggsave(
  filename = "/home/jupyter/workspaces/cypphenoconversion/data/Figures/Supp_DrugSpecific_All_Genetic_wMod.png",
  plot     = p1,
  width    = 8,
  height   = 8,
  dpi      = 600
)


In [ ]:
#-- All ancestries and raw genotype effects for those without modualtors in the any-follow-up window
plot_data <- all_df %>%
  filter(Model == "CYP_Main_raw_NoMod") %>%
  filter(str_detect(variable, "CYP")) %>%
  filter(Ancestry == "All") %>%
  mutate(Drug = gsub("Serotonin_Modulator", "SM", Drug)) %>%
  filter(Window == "any_followup")  %>%
  filter(SexGroup == "All")

n_drugs <- length(unique(plot_data$Drug))

plot_data<- plot_data %>%
        mutate(Sig_FDR_drug = 
           case_when(p_fdr_drug < 0.05 ~ "Sig", 
                     p_fdr_drug >= 0.05 ~ "NotSig",
                    is.na(p_fdr_drug) ~ "NotSig",
                    TRUE ~ NA_character_),
          Sig_FDR = 
          case_when(p_fdr < 0.05 ~ "Sig", 
                     p_fdr >= 0.05 ~ "NotSig",
                    is.na(p_fdr) ~ "NotSig",
                    TRUE ~ NA_character_),
          Sig_Nom = 
          case_when(p_value < 0.05 & p_fdr_drug >= 0.05 ~ TRUE, 
                     p_value >= 0.05 ~ FALSE,
                    is.na(p_value) ~ FALSE,
                    TRUE ~ NA))

plot_data$SigColour <- ifelse(
  plot_data$Sig_FDR_drug == "Sig", 
  "Sig", 
  "NotSig"
)

plot_data$variable <- factor(plot_data$variable,
   levels = c(
      "CYP2B6_P/I", "CYP2B6_N", "CYP2B6_U",
      "CYP2C19_P","CYP2C19_I","CYP2C19_R","CYP2C19_N","CYP2C19_U",
      "CYP2D6_P","CYP2D6_I","CYP2D6_N","CYP2D6_U"
))

plot_data <- plot_data %>%
    mutate(Gene = case_when(
    str_detect(variable, "CYP2D6") ~ "CYP2D6",
    str_detect(variable, "CYP2B6") ~ "CYP2B6",
    TRUE ~ "CYP2C19"))


fill_colors <- c(
  "CYP2D6" = "#c4253d",
  "CYP2C19" = "#FD8D3C",
  "CYP2B6" = "#e9b43b"
)
plot_data <- plot_data %>%
    mutate(Drug = paste0(Drug, "\nN=", N_Total, " (", round(N_Cases/N_Total*100, 0), "%)"))

In [ ]:
plot_data <- plot_data %>%
  mutate(
    SigFill = if_else(Sig_FDR_drug == "Sig", Gene, "NotSig")
  )

fill_colors <- c(
  "CYP2D6"  = "#c4253d",
  "CYP2C19" = "#FD8D3C",
  "CYP2B6"  = "#e9b43b"
)

pos <- position_dodge(width = 0.5)

p1 <- ggplot(plot_data, aes(x = odds_ratio, y = variable)) +
  geom_vline(xintercept = 1, linetype = "dashed") +
  geom_errorbarh(
    aes(xmin = lower_95, xmax = upper_95, colour = Gene),
    position = pos
  ) +
  geom_point(
    aes(colour = Gene, fill = SigFill),
    position = pos,
    shape  = 21,
    size   = 2.5,
    stroke = 0.7
  ) +

  labs(
    x = "Odds ratio (95% CI, log-scale)",
    y = "Genetic metabolizer status"
  ) +
  theme_minimal() +
  theme(
    strip.background = element_rect(fill="grey92", colour=NA),
    strip.text.y = element_text(angle = 0),
    strip.text.x = element_text(angle = 0),
    axis.text    = element_text(color = "black"),
    strip.text   = element_text(face = "bold"),
    axis.title = element_text(face = "bold"),
    legend.title = element_text(face = "bold"),
    legend.position = "bottom"
  ) +
  scale_color_manual(values = fill_colors, name = "Gene") +
  scale_fill_manual(
    values = c(fill_colors, NotSig = "white"),
    guide  = "none"
  ) +

  facet_wrap(Drug ~ ., ncol = 3) +
  scale_x_log10()

p1


In [ ]:
# Save as PNG
ggsave(
  filename = "/home/jupyter/workspaces/cypphenoconversion/data/Figures/Supp_DrugSpecific_All_Genetic_noMod.png",
  plot     = p1,
  width    = 8,
  height   = 8,
  dpi      = 600
)

In [ ]:
# Cov all no CYP
library(dplyr)
library(stringr)
library(forcats)
library(ggplot2)
library(readxl)

all_df <- read_excel("/home/jupyter/workspaces/cypphenoconversion/data/All_Results_CYP.xlsx", sheet = "DrugSpecific_Covariates")

pos <- position_dodge(width = 0.5)

plot_data <- all_df %>%
  filter(Model == "CovAll_NoCYP") %>%
  filter(!str_detect(variable, "(Intercept)")) %>%
  filter(Ancestry == "All") %>%
  mutate(Drug = gsub("Serotonin_Modulator", "SM", Drug)) %>%
  filter(Window == "any_followup")  %>%
  filter(SexGroup == "All")

n_drugs <- length(unique(plot_data$Drug))

plot_data<- plot_data %>%
        mutate(Sig_FDR_drug = 
           case_when(p_fdr_drug < 0.05 ~ "Sig", 
                     p_fdr_drug >= 0.05 ~ "NotSig",
                    is.na(p_fdr_drug) ~ "NotSig",
                    TRUE ~ NA_character_),
          Sig_FDR = 
          case_when(p_fdr < 0.05 ~ "Sig", 
                     p_fdr >= 0.05 ~ "NotSig",
                    is.na(p_fdr) ~ "NotSig",
                    TRUE ~ NA_character_),
          Sig_Nom = 
          case_when(p_value < 0.05 ~ TRUE, 
                     p_value >= 0.05 ~ FALSE,
                    is.na(p_value) ~ FALSE,
                    TRUE ~ NA))

plot_data$SigColour <- ifelse(
  plot_data$Sig_FDR_drug == "Sig", 
  "Sig", 
  "NotSig"
)

plot_data <- plot_data %>%
    mutate(Covariate = case_when(
    str_detect(variable, "cyp2d6") ~ "CYP2D6 Modulator",
    str_detect(variable, "cyp2b6") ~ "CYP2B6 Modulator",
    str_detect(variable, "cyp2c19") ~ "CYP2C19 Modulator",
    str_detect(variable, "neg")  ~ "Negative Control",
    TRUE ~ "Other"))

plot_data <- plot_data %>%
    filter(Covariate != "Other")

plot_data <- plot_data %>%
    mutate(Drug = paste0(Drug, "\nN=", N_Total, " (", round(N_Cases/N_Total*100, 0), "%)"))

plot_data <- plot_data %>%
    mutate(variable = 
          case_when(variable == "Education_levelTwelve Or GED" ~ "Education:Twelve/GED",
                   variable == "Education_levelCollege Undergraduate" ~ "Education:Undergraduate",
                   variable == "Education_levelCollege Postgraduate" ~ "Education:Postgraduate",
                   TRUE ~ variable))
unique(plot_data$variable)
plot_data$variable = factor(plot_data$variable, levels = c("age_at_first_AD", "sex_at_birthMale", "Income_quantitative","num_prior_AD",
                                               "Education:Twelve/GED", "Education:Undergraduate", 
                                              "ancestry_predamr",  "ancestry_predeur", "ancestry_predOther", 
                                               "Education:Postgraduate", "prav_neg_cont", "rosu_neg_cont", "cyp2b6_weak_mod_ind",
                                              "cyp2b6_weak_inhib", "cyp2b6_mod_strong_inhib",  "cyp2c19_weak_mod_ind", 
                                              "cyp2c19_weak_inhib", "cyp2c19_mod_inhib", 
                                              "cyp2c19_strong_inhib", "cyp2d6_weak_inhib", 
                                              "cyp2d6_mod_strong_inhib"))

In [ ]:
# Fix the missing # in Other
fill_colors <- c(
  "CYP2D6 Modulator"  = "#2c7fb8",
  "CYP2C19 Modulator" = "#984ea3",
  "CYP2B6 Modulator"  = "#1a9850",
  "Negative Control"  = "#d95f02",
  "Other" = "#8b0000"
)

# Create a fill variable that matches your color scale keys
plot_data <- plot_data %>%
    mutate(
    FillVar = case_when(
    Covariate %in% c("CYP2D6 Modulator", "CYP2C19 Modulator", "CYP2B6 Modulator") & Sig_FDR_drug == "Sig" ~ Covariate,
    Covariate == "Negative Control" & Sig_Nom ~ Covariate,
    TRUE ~ "NotSig"))

plot_data$Drug = factor(plot_data$Drug, levels = c("SSRI:sertraline\nN=6626 (31%)",
                                              "SNRI:venlafaxine\nN=3078 (34%)",
                                              "TCA:amitriptyline\nN=2680 (27%)",
                                              "SSRI:escitalopram\nN=4432 (26%)",
                                              "SNRI:duloxetine\nN=4101 (26%)",
                                              "NDRI:bupropion\nN=4842 (29%)",
                                              "NaSSA:mirtazapine\nN=1194 (39%)",
                                              "SSRI:citalopram\nN=3877 (31%)",
                                              "SSRI:fluoxetine\nN=3663 (29%)",
                                              "SSRI:paroxetine\nN=1462 (27%)",
                                              "SARI:trazodone\nN=3813 (34%)",
                                              "TCA:nortriptyline\nN=1495 (27%)"))

# Then in your plot:
p1 <- ggplot(plot_data, aes(x = odds_ratio, y = variable)) +
  geom_vline(xintercept = 1, linetype = "dashed") +
  geom_errorbarh(
    aes(xmin = lower_95, xmax = upper_95, color = Covariate),
    position = pos
  ) +
  geom_point(
    aes(color = Covariate, fill = FillVar),
    position = pos,
    shape = 21, 
    size = 2.5,
    stroke = 0.7
  ) +
  labs(
    x = "Odds ratio (95% CI)",
    y = "CYP Modulators and Negative Controls"
  ) +
  theme_minimal() +
  theme(strip.background = element_rect(fill="grey92", colour=NA),
    strip.text.y = element_text(angle = 0),
    strip.text.x = element_text(angle = 0),
    axis.text = element_text(color = "black"),
    strip.text = element_text(face = "bold"),
    axis.title = element_text(face = "bold"),
    legend.title = element_text(face = "bold"),
    legend.position = "bottom"
  ) +
  scale_color_manual(values = fill_colors, name = "Exposure") +
  scale_fill_manual(
    values = c(fill_colors, NotSig = "white"),
    guide = "none"
  ) +
  facet_wrap(Drug ~ ., ncol = 3) +
  scale_x_continuous(limits = c(0, 3), oob = scales::squish)

p1

In [ ]:
# Save as PNG
ggsave(
  filename = "/home/jupyter/workspaces/cypphenoconversion/data/Figures/Main_DrugSpecific_All_Cov.png",
  plot     = p1,
  width    = 9,
  height   = 9,
  dpi      = 600
)


In [ ]:
# Cov all no CYP
library(dplyr)
library(stringr)
library(forcats)
library(ggplot2)
library(readxl)

all_df <- read_excel("/home/jupyter/workspaces/cypphenoconversion/data/All_Results_CYP.xlsx", sheet = "DrugSpecific_Covariates")

pos <- position_dodge(width = 0.5)

plot_data <- all_df %>%
  filter(Model == "CovAll_NoCYP") %>%
  filter(!str_detect(variable, "(Intercept)")) %>%
  filter(Ancestry == "All") %>%
  mutate(Drug = gsub("Serotonin_Modulator", "SM", Drug)) %>%
  filter(Window == "any_followup")  %>%
  filter(SexGroup == "All")

n_drugs <- length(unique(plot_data$Drug))

plot_data<- plot_data %>%
        mutate(Sig_FDR_drug = 
           case_when(p_fdr_drug < 0.05 ~ "Sig", 
                     p_fdr_drug >= 0.05 ~ "NotSig",
                    is.na(p_fdr_drug) ~ "NotSig",
                    TRUE ~ NA_character_),
          Sig_FDR = 
          case_when(p_fdr < 0.05 ~ "Sig", 
                     p_fdr >= 0.05 ~ "NotSig",
                    is.na(p_fdr) ~ "NotSig",
                    TRUE ~ NA_character_),
          Sig_Nom = 
          case_when(p_value < 0.05 ~ TRUE, 
                     p_value >= 0.05 ~ FALSE,
                    is.na(p_value) ~ FALSE,
                    TRUE ~ NA))

plot_data$SigColour <- ifelse(
  plot_data$Sig_FDR_drug == "Sig", 
  "Sig", 
  "NotSig"
)

plot_data <- plot_data %>%
    mutate(Covariate = case_when(
    str_detect(variable, "cyp2d6") ~ "CYP2D6 Modulator",
    str_detect(variable, "cyp2b6") ~ "CYP2B6 Modulator",
    str_detect(variable, "cyp2c19") ~ "CYP2C19 Modulator",
    str_detect(variable, "neg")  ~ "Negative Control",
    TRUE ~ "Other"))

plot_data <- plot_data %>%
    filter(Covariate == "Other")

plot_data <- plot_data %>%
    mutate(Drug = paste0(Drug, "\nN=", N_Total, " (", round(N_Cases/N_Total*100, 0), "%)"))

plot_data <- plot_data %>%
    mutate(variable = 
          case_when(variable == "Education_levelTwelve Or GED" ~ "Education:Twelve/GED",
                   variable == "Education_levelCollege Undergraduate" ~ "Education:Undergraduate",
                   variable == "Education_levelCollege Postgraduate" ~ "Education:Postgraduate",
                   TRUE ~ variable))
unique(plot_data$variable)
plot_data$variable = factor(plot_data$variable, levels = c("age_at_first_AD", "sex_at_birthMale", "Income_quantitative","num_prior_AD",
                                               "Education:Twelve/GED", "Education:Undergraduate", 
                                              "ancestry_predamr",  "ancestry_predeur", "ancestry_predOther", 
                                               "Education:Postgraduate", "prav_neg_cont", "rosu_neg_cont", "cyp2b6_weak_mod_ind",
                                              "cyp2b6_weak_inhib", "cyp2b6_mod_strong_inhib",  "cyp2c19_weak_mod_ind", 
                                              "cyp2c19_weak_inhib", "cyp2c19_mod_inhib", 
                                              "cyp2c19_strong_inhib", "cyp2d6_weak_inhib", "cyp2d6_mod_inhib", 
                                              "cyp2d6_mod_strong_inhib"))

In [ ]:
# Fix the missing # in Other
fill_colors <- c(
  "CYP2D6 Modulator"  = "#2c7fb8",
  "CYP2C19 Modulator" = "#984ea3",
  "CYP2B6 Modulator"  = "#1a9850",
  "Negative Control"  = "#d95f02",
  "Other" = "#8b0000"
)

# Create a fill variable that matches your color scale keys
plot_data <- plot_data %>%
    mutate(
    FillVar = case_when(
    Covariate %in% c("CYP2D6 Modulator", "CYP2C19 Modulator", "CYP2B6 Modulator") & Sig_FDR_drug == "Sig" ~ Covariate,
    Covariate == "Other" & Sig_Nom ~ Covariate,
    TRUE ~ "NotSig"))

# Then in your plot:
p1 <- ggplot(plot_data, aes(x = odds_ratio, y = variable)) +
  geom_vline(xintercept = 1, linetype = "dashed") +
  geom_errorbarh(
    aes(xmin = lower_95, xmax = upper_95, color = Covariate),
    position = pos
  ) +
  geom_point(
    aes(color = Covariate, fill = FillVar),
    position = pos,
    shape = 21, 
    size = 2.5,
    stroke = 0.7
  ) +
  labs(
    x = "Odds ratio (95% CI)",
    y = "Covariates"
  ) +
  theme_minimal() +
  theme(strip.background = element_rect(fill="grey92", colour=NA),
    strip.text.y = element_text(angle = 0),
    strip.text.x = element_text(angle = 0),
    axis.text = element_text(color = "black"),
    strip.text = element_text(face = "bold"),
    axis.title = element_text(face = "bold"),
    legend.title = element_text(face = "bold"),
    legend.position = "bottom"
  ) +
  scale_color_manual(values = fill_colors, name = "Exposure") +
  scale_fill_manual(
    values = c(fill_colors, NotSig = "white"),
    guide = "none"
  ) +
  facet_wrap(Drug ~ ., ncol = 4) +
  scale_x_continuous(limits = c(0, 3), oob = scales::squish)

p1

In [ ]:
#-- Comparison across ancestries
all_df <- read_excel("/home/jupyter/workspaces/cypphenoconversion/data/All_Results_CYP.xlsx", sheet = "DrugSpecific_All_Ancestries")
all_df3 <- read_excel("/home/jupyter/workspaces/cypphenoconversion/data/All_Results_CYP.xlsx", sheet = "DrugSpecific_European")
df <- rbind(all_df, all_df3)
plot_data <- df %>%
  filter(str_detect(variable, "CYP")) %>%
  filter(Window == "any_followup") %>%
  mutate(Drug = gsub("Serotonin_Modulator", "SM", Drug))  %>%
  filter(SexGroup == "All") %>%
  filter(Model == "CYP_Main_adj")


n_drugs <- length(unique(plot_data$Drug))

plot_data<- plot_data %>%
        mutate(Sig_FDR_drug = 
           case_when(p_fdr_drug < 0.05 ~ "Sig", 
                     p_fdr_drug >= 0.05 ~ "NotSig",
                    is.na(p_fdr_drug) ~ "NotSig",
                    TRUE ~ NA_character_),
          Sig_FDR = 
          case_when(p_fdr < 0.05 ~ "Sig", 
                     p_fdr >= 0.05 ~ "NotSig",
                    is.na(p_fdr) ~ "NotSig",
                    TRUE ~ NA_character_),
          Sig_Nom = 
          case_when(p_value < 0.05 & p_fdr_drug >= 0.05 ~ TRUE, 
                     p_value >= 0.05 ~ FALSE,
                    is.na(p_value) ~ FALSE,
                    TRUE ~ NA))

plot_data$SigColour <- ifelse(
  plot_data$Sig_FDR_drug == "Sig", 
  "Sig", 
  "NotSig"
)


plot_data <- plot_data %>%
    mutate(Gene = case_when(
    str_detect(variable, "CYP2D6") ~ "CYP2D6",
    str_detect(variable, "CYP2B6") ~ "CYP2B6",
    TRUE ~ "CYP2C19")) %>%
    filter(Drug %in% c("TCA:amitriptyline", "SSRI:sertraline", "NaSSA:mirtazapine", 
                       "NDRI:bupropion", "SNRI:duloxetine", "SNRI:venlafaxine",
                      "SSRI:escitalopram"))

fill_colors <- c(
  "European" = "#07A0C3",
  "All" = "#EC7505"
)


In [ ]:
plot_data <- plot_data %>%
  mutate(
    SigFill = if_else(Sig_FDR_drug == "Sig", Ancestry, "NotSig")
  )
pos <- position_dodge(width = 0.5)

p1 <- ggplot(plot_data, aes(x = odds_ratio, y = variable, group = Ancestry)) + 
  geom_vline(xintercept = 1, linetype = "dashed") +
  geom_errorbarh(
    aes(xmin = lower_95, xmax = upper_95, colour = Ancestry),
    position = pos
  ) +
  geom_point(
    aes(fill = SigFill, colour = Ancestry),
    position = pos,
    shape  = 21,
    size   = 2.5,
    stroke = 0.7
  ) +
  labs(
    x = "Odds ratio (95% CI, log-scale)",
    y = "Phenoconversion-adjusted metabolizer status"
  ) +
  theme_minimal() +
  theme(
    strip.background = element_rect(fill="grey92", colour=NA),
    strip.text.y = element_text(angle = 0),
    strip.text.x = element_text(angle = 0),
    axis.text    = element_text(color = "black"),
    strip.text   = element_text(face = "bold"),
    axis.title = element_text(face = "bold"),
    legend.title = element_text(face = "bold"),
    legend.position = "bottom"
  ) +
  scale_color_manual(values = fill_colors, name = "Ancestry") +
  scale_fill_manual(
    values = c(fill_colors, NotSig = "white"),
    guide  = "none"
  ) +
  facet_wrap(Drug ~ ., nrow = 2)
p1


In [ ]:
# Save as PNG
ggsave(
  filename = "/home/jupyter/workspaces/cypphenoconversion/data/Figures/Supp_DrugSpecific_Ancestry_Sens.png",
  plot     = p1,
  width    = 11,
  height   = 7,
  dpi      = 600
)

In [ ]:
#-- Comparison across sex strata for sig results
all_df4 <- read_excel("/home/jupyter/workspaces/cypphenoconversion/data/All_Results_CYP.xlsx", sheet = "DrugSpecific_SexStratified")
df <- rbind(all_df, all_df4)
plot_data <- df %>%
  filter(str_detect(variable, "CYP")) %>%
  filter(Ancestry == "All") %>%
  mutate(Drug = gsub("Serotonin_Modulator", "SM", Drug))  %>%
  filter(Model == "CYP_Main_adj") %>%
  filter(Window == "any_followup")
unique(plot_data$SexGroup)

n_drugs <- length(unique(plot_data$Drug))

plot_data<- plot_data %>%
        mutate(Sig_FDR_drug = 
           case_when(p_fdr_drug < 0.05 ~ "Sig", 
                     p_fdr_drug >= 0.05 ~ "NotSig",
                    is.na(p_fdr_drug) ~ "NotSig",
                    TRUE ~ NA_character_),
          Sig_FDR = 
          case_when(p_fdr < 0.05 ~ "Sig", 
                     p_fdr >= 0.05 ~ "NotSig",
                    is.na(p_fdr) ~ "NotSig",
                    TRUE ~ NA_character_),
          Sig_Nom = 
          case_when(p_value < 0.05 & p_fdr_drug >= 0.05 ~ TRUE, 
                     p_value >= 0.05 ~ FALSE,
                    is.na(p_value) ~ FALSE,
                    TRUE ~ NA))

plot_data$SigColour <- ifelse(
  plot_data$Sig_FDR_drug == "Sig", 
  "Sig", 
  "NotSig"
)


plot_data <- plot_data %>%
    mutate(Gene = case_when(
    str_detect(variable, "CYP2D6") ~ "CYP2D6",
    str_detect(variable, "CYP2B6") ~ "CYP2B6",
    TRUE ~ "CYP2C19")) %>%
    filter(Drug %in% c("TCA:amitriptyline", "SSRI:sertraline", "NaSSA:mirtazapine", 
                   "NDRI:bupropion", "SNRI:duloxetine", "SNRI:venlafaxine",
                  "SSRI:escitalopram"))


fill_colors <- c(
  "All" = "#EC7505",
  "Female" = "#FF006E",
  "Male" = "#5C95FF"
)
unique(plot_data$SexGroup)


In [ ]:
# add a fill variable: gene colour if Sig, "NotSig" otherwise
plot_data <- plot_data %>%
  mutate(
    SigFill = if_else(Sig_FDR_drug == "Sig", SexGroup, "NotSig")
  )
pos <- position_dodge(width = 0.5)

p2 <- ggplot(plot_data, aes(x = odds_ratio, y = variable, group = SexGroup)) + 
  geom_vline(xintercept = 1, linetype = "dashed") +
  geom_errorbarh(
    aes(xmin = lower_95, xmax = upper_95, colour = SexGroup),
    position = pos
  ) +
  geom_point(
    aes(fill = SigFill, colour = SexGroup),
    position = pos,
    shape  = 21,
    size   = 2.5,
    stroke = 0.7
  ) +
  labs(
    x = "Odds ratio (95% CI, log-scale)",
    y = NULL
  ) +
  theme_minimal() +
  scale_x_continuous(limits = c(0, 3), oob = scales::squish) +
  theme(
    strip.background = element_rect(fill="grey92", colour=NA),
    strip.text.y = element_text(angle = 0),
    strip.text.x = element_text(angle = 0),
    axis.text    = element_text(color = "black"),
    strip.text   = element_text(face = "bold"),
    axis.title = element_text(face = "bold"),
    legend.title = element_text(face = "bold"),
    legend.position = "bottom"
  ) +
  scale_color_manual(values = fill_colors, name = "Sex") +
  scale_fill_manual(
    values = c(fill_colors, NotSig = "white"),
    guide  = "none"
  ) +
  facet_wrap(Drug ~ ., nrow = 2)
p2


In [ ]:
# Save as PNG
ggsave(
  filename = "/home/jupyter/workspaces/cypphenoconversion/data/Figures/Supp_DrugSpecific_Sex_Sens.png",
  plot     = p2,
  width    = 11,
  height   = 8,
  dpi      = 600
)

In [ ]:
#-- Comparison across CYP modulator windows
all_df4 <- read_excel("/home/jupyter/workspaces/cypphenoconversion/data/All_Results_CYP.xlsx", sheet = "DrugSpecific_TemporalWindows")
df <- rbind(all_df, all_df4)
plot_data <- df %>%
  filter(str_detect(variable, "CYP")) %>%
  filter(Ancestry == "All") %>%
  mutate(Drug = gsub("Serotonin_Modulator", "SM", Drug))  %>%
  filter(Model == "CYP_Main_adj") %>%
  filter(SexGroup == "All")

n_drugs <- length(unique(plot_data$Drug))

plot_data<- plot_data %>%
        mutate(Sig_FDR_drug = 
           case_when(p_fdr_drug < 0.05 ~ "Sig", 
                     p_fdr_drug >= 0.05 ~ "NotSig",
                    is.na(p_fdr_drug) ~ "NotSig",
                    TRUE ~ NA_character_),
          Sig_FDR = 
          case_when(p_fdr < 0.05 ~ "Sig", 
                     p_fdr >= 0.05 ~ "NotSig",
                    is.na(p_fdr) ~ "NotSig",
                    TRUE ~ NA_character_),
          Sig_Nom = 
          case_when(p_value < 0.05 & p_fdr_drug >= 0.05 ~ TRUE, 
                     p_value >= 0.05 ~ FALSE,
                    is.na(p_value) ~ FALSE,
                    TRUE ~ NA))

plot_data$SigColour <- ifelse(
  plot_data$Sig_FDR_drug == "Sig", 
  "Sig", 
  "NotSig"
)


plot_data <- plot_data %>%
    mutate(Gene = case_when(
    str_detect(variable, "CYP2D6") ~ "CYP2D6",
    str_detect(variable, "CYP2B6") ~ "CYP2B6",
    TRUE ~ "CYP2C19")) %>%
    filter(Drug %in% c("TCA:amitriptyline", "SSRI:sertraline", "NaSSA:mirtazapine", 
                   "NDRI:bupropion", "SNRI:duloxetine", "SNRI:venlafaxine",
                  "SSRI:escitalopram"))

fill_colors <- c(
  "any_followup" = "#EC7505",
  "baseline_only" = "#5ADBFF",
  "post_index_only" = "#820B8A"
)


In [ ]:
# add a fill variable: gene colour if Sig, "NotSig" otherwise
plot_data <- plot_data %>%
  mutate(
    SigFill = if_else(Sig_FDR_drug == "Sig", Window, "NotSig")
  )
pos <- position_dodge(width = 0.5)

p3 <- ggplot(plot_data, aes(x = odds_ratio, y = variable, group = Window)) + 
  geom_vline(xintercept = 1, linetype = "dashed") +
  geom_errorbarh(
    aes(xmin = lower_95, xmax = upper_95, colour = Window),
    position = pos
  ) +
  geom_point(
    aes(fill = SigFill, colour = Window),
    position = pos,
    shape  = 21,
    size   = 2.5,
    stroke = 0.7
  ) +
  labs(
    x = "Odds ratio (95% CI, log-scale)",
    y = NULL
  ) +
  theme_minimal() +
  theme(
    strip.background = element_rect(fill="grey92", colour=NA),
    strip.text.y = element_text(angle = 0),
    strip.text.x = element_text(angle = 0),
    axis.text    = element_text(color = "black"),
    strip.text   = element_text(face = "bold"),
    axis.title = element_text(face = "bold"),
    legend.title = element_text(face = "bold"),
    legend.position = "bottom"
  ) +
  scale_color_manual(values = fill_colors, name = "Window") +
  scale_fill_manual(
    values = c(fill_colors, NotSig = "white"),
    guide  = "none"
  ) +
  facet_wrap(Drug ~ ., nrow=2)
p3


In [ ]:
# Save as PNG
ggsave(
  filename = "/home/jupyter/workspaces/cypphenoconversion/data/Figures/Supp_DrugSpecific_Window_Sens.png",
  plot     = p3,
  width    = 11,
  height   = 8,
  dpi      = 600
)


In [ ]:
library(dplyr)
library(lubridate)
library(readr)

# -----------------------------------------------------------------------------
# Formatting helpers
# -----------------------------------------------------------------------------

# Format a number to `sf` significant figures, preserving trailing zeros.
# e.g. format_sigfig(0.045, 2) -> "0.045"
#      format_sigfig(0.5,   2) -> "0.50"
#      format_sigfig(12.3,  2) -> "12"
#      format_sigfig(87.6,  2) -> "88"
format_sigfig <- function(x, sf = 2) {
  if (is.na(x)) return(NA_character_)
  if (x == 0)   return("0")
  rounded <- signif(x, sf)
  dp <- max(0, sf - 1 - floor(log10(abs(rounded))))
  formatC(rounded, format = "f", digits = dp)
}

# Round UP to `sf` significant figures (for suppression bounds, so the
# reported "<X%" is guaranteed to be a true upper limit on the underlying value).
ceiling_sigfig <- function(x, sf = 2) {
  if (is.na(x) || x == 0) return(x)
  mag <- 10^(floor(log10(abs(x))) - sf + 1)
  ceiling(x / mag) * mag
}

# -----------------------------------------------------------------------------
# AoU <20 suppression helper
# Returns list(n, pct) as character strings, per the AoU Data and Statistics
# Dissemination Policy:
#   - n = 0          -> "0"
#   - 0 < n < 20     -> "<20" and bounded "<X%" upper limit (sf-rounded UP)
#   - n >= 20        -> exact n and % to `sf` significant figures
# -----------------------------------------------------------------------------
aou_cell <- function(n, total, sf = 2) {
  if (is.na(n) || is.na(total) || total == 0) {
    return(list(n = "—", pct = "—"))
  }
  if (n == 0) {
    return(list(n = "—", pct = "—"))
  }
  if (n < 20) {
    bound <- ceiling_sigfig(100 * 20 / total, sf)
    return(list(n   = "<20",
                pct = paste0("<", format_sigfig(bound, sf))))
  }
  list(
    n   = as.character(n),
    pct = format_sigfig(100 * n / total, sf)
  )
}

# -----------------------------------------------------------------------------
# Parameters
# -----------------------------------------------------------------------------
cyp_genes   <- c("cyp2d6", "cyp2c19", "cyp2b6")
cyp_windows <- c("any_followup", "baseline_only", "concom_only")

gene_modulators <- list(
  cyp2d6  = c("strong_inhibitors", "moderate_inhibitors", "weak_inhibitors"),
  cyp2c19 = c("strong_inhibitors", "moderate_inhibitors", "weak_inhibitors",
              "moderate_inducers", "weak_inducers"),
  cyp2b6  = c("strong_inhibitors", "moderate_inhibitors", "weak_inhibitors",
              "moderate_inducers", "weak_inducers")
)

# -----------------------------------------------------------------------------
# Build summaries
# -----------------------------------------------------------------------------
exposure_summary_list         <- list()  # AoU-suppressed (character)
exposure_summary_numeric_list <- list()  # raw numeric, INTERNAL only

for (drug in drugs) {

  print(paste("Computing modulator exposure for:", drug))

  drug_data <- outcome %>%
    filter(drug.treatment == drug) %>%
    left_join(pheno, by = "person_id") %>%
    mutate(age_at_first_AD = interval(date_of_birth, DateFirstDispense) %/% years(1))

  drug_data_unique <- drug_data %>%
    group_by(person_id) %>%
    filter(FirstDispense.treatment == min(FirstDispense.treatment)) %>%
    slice(1) %>%
    ungroup() %>%
    filter(has_any_AD_modulator_ever == 0L)

  n_total <- nrow(drug_data_unique)

  if (n_total == 0) {
    message("  No data for ", drug); next
  }

  if (n_total < 20) {
    warning("Drug total n_total < 20 for ", drug,
            " -- suppress the row entirely or collapse before reporting")
  }

  for (gene in cyp_genes) {
    for (window in cyp_windows) {

      modulators <- gene_modulators[[gene]]

      summary_row <- data.frame(
        Drug = drug, Gene = toupper(gene), Window = window,
        N = as.character(n_total),
        stringsAsFactors = FALSE
      )
      numeric_row <- data.frame(
        Drug = drug, Gene = toupper(gene), Window = window,
        N = n_total,
        stringsAsFactors = FALSE
      )

      any_inhib_cols <- c()
      any_ind_cols   <- c()

      for (mod in modulators) {
        col_name <- paste0(window, "_", gene, "_", mod)

        if (!col_name %in% names(drug_data_unique)) {
          message("  Column not found: ", col_name); next
        }

        n_exposed   <- sum(drug_data_unique[[col_name]] == 1, na.rm = TRUE)
        pct_exposed <- signif(100 * n_exposed / n_total, 2)
        cell        <- aou_cell(n_exposed, n_total)

        mod_clean <- gsub("_", " ", mod)
        mod_clean <- tools::toTitleCase(mod_clean)
        mod_clean <- gsub(" ", "_", mod_clean)

        # Suppressed (manuscript)
        summary_row[[paste0("N_",   mod_clean)]] <- cell$n
        summary_row[[paste0("Pct_", mod_clean)]] <- cell$pct
        # Numeric (internal)
        numeric_row[[paste0("N_",   mod_clean)]] <- n_exposed
        numeric_row[[paste0("Pct_", mod_clean)]] <- pct_exposed

        if (grepl("inhibitor", mod)) {
          any_inhib_cols <- c(any_inhib_cols, col_name)
        } else if (grepl("inducer", mod)) {
          any_ind_cols <- c(any_ind_cols, col_name)
        }
      }

      # Any inhibitor
      if (length(any_inhib_cols) > 0) {
        any_inhib <- rowSums(drug_data_unique[, any_inhib_cols, drop = FALSE] == 1,
                             na.rm = TRUE) > 0
        n_any   <- sum(any_inhib)
        pct_any <- signif(100 * mean(any_inhib), 2)
        cell    <- aou_cell(n_any, n_total)

        summary_row$N_Any_Inhibitor   <- cell$n
        summary_row$Pct_Any_Inhibitor <- cell$pct
        numeric_row$N_Any_Inhibitor   <- n_any
        numeric_row$Pct_Any_Inhibitor <- pct_any
      }

      # Any inducer
      if (length(any_ind_cols) > 0) {
        any_ind <- rowSums(drug_data_unique[, any_ind_cols, drop = FALSE] == 1,
                           na.rm = TRUE) > 0
        n_any   <- sum(any_ind)
        pct_any <- signif(100 * mean(any_ind), 2)
        cell    <- aou_cell(n_any, n_total)

        summary_row$N_Any_Inducer   <- cell$n
        summary_row$Pct_Any_Inducer <- cell$pct
        numeric_row$N_Any_Inducer   <- n_any
        numeric_row$Pct_Any_Inducer <- pct_any
      }

      key <- paste(drug, gene, window, sep = "_")
      exposure_summary_list[[key]]         <- summary_row
      exposure_summary_numeric_list[[key]] <- numeric_row
    }
  }
}

# -----------------------------------------------------------------------------
# Combine
# -----------------------------------------------------------------------------
exposure_summary_df         <- bind_rows(exposure_summary_list)
exposure_summary_numeric_df <- bind_rows(exposure_summary_numeric_list)

exposure_summary_df <- exposure_summary_df %>%
  mutate(Window = case_when(
    Window == "concom_only" ~ "post_index_only",
    TRUE ~ Window
  )) %>%
  mutate(across(starts_with(c("N_", "Pct_")),
                ~ ifelse(is.na(.x), "—", .x)))

# Quick views
print(exposure_summary_df)


In [ ]:
write_csv(exposure_summary_numeric_df,
          "/home/jupyter/workspaces/cypphenoconversion/data/modulator_prevalence_INTERNAL_numeric.csv")

wb <- loadWorkbook("/home/jupyter/workspaces/cypphenoconversion/data/All_Results_CYP.xlsx")

removeWorksheet(wb, "DrugSpecific_Modulator_Prev")
addWorksheet(wb, "DrugSpecific_Modulator_Prev")
writeData(wb, "DrugSpecific_Modulator_Prev", exposure_summary_df)

saveWorkbook(wb, "/home/jupyter/workspaces/cypphenoconversion/data/All_Results_CYP.xlsx", overwrite=TRUE)


In [ ]:
# Compute inhibitor AND inducer exposure by drug, sex, and CYP gene
modulator_exposure_summary_sex <- list()

cyp_genes <- c("cyp2d6", "cyp2c19", "cyp2b6")

for (drug in drugs) {
  
  print(paste("Computing modulator exposure for:", drug))
  
  # Get all treatment episodes for this drug
  drug_data <- outcome %>%
    filter(drug.treatment == drug) %>%
    left_join(pheno, by = "person_id") %>%
    mutate(
      age_at_first_AD = interval(date_of_birth, DateFirstDispense) %/% years(1)
    ) %>%
    filter(age_at_first_AD >= 12)
  
  # Remove duplicates - one row per person
  drug_data_unique <- drug_data %>%
    group_by(person_id) %>%
    filter(FirstDispense.treatment == min(FirstDispense.treatment)) %>%
    slice(1) %>%
    ungroup() %>%
    filter(has_any_AD_modulator_ever == 0L)
  
  if (nrow(drug_data_unique) == 0) {
    message("  No data for ", drug)
    next
  }
  
  #=========================
  # Loop over CYP genes, windows, and stratify by sex
  #=========================
  for (gene in cyp_genes) {
    for (window in c("any_followup", "baseline_only", "concom_only")) {
      
      # Get window-specific column names - INHIBITORS
      strong_inhib_col <- paste0(window, "_", gene, "_strong_inhibitors")
      moderate_inhib_col <- paste0(window, "_", gene, "_moderate_inhibitors")
      weak_inhib_col <- paste0(window, "_", gene, "_weak_inhibitors")
      
      # Get window-specific column names - INDUCERS
      strong_ind_col <- paste0(window, "_", gene, "_strong_inducers")
      moderate_ind_col <- paste0(window, "_", gene, "_moderate_inducers")
      weak_ind_col <- paste0(window, "_", gene, "_weak_inducers")
      
      # Check which columns actually exist in the data
      has_strong_inhib <- strong_inhib_col %in% names(drug_data_unique)
      has_moderate_inhib <- moderate_inhib_col %in% names(drug_data_unique)
      has_weak_inhib <- weak_inhib_col %in% names(drug_data_unique)
      has_strong_ind <- strong_ind_col %in% names(drug_data_unique)
      has_moderate_ind <- moderate_ind_col %in% names(drug_data_unique)
      has_weak_ind <- weak_ind_col %in% names(drug_data_unique)
      
      # Skip if no columns exist for this gene/window combination
      has_any_inhib <- has_strong_inhib | has_moderate_inhib | has_weak_inhib
      has_any_ind <- has_strong_ind | has_moderate_ind | has_weak_ind
      if (!has_any_inhib & !has_any_ind) next
      
      # Calculate by sex
      sex_summary <- drug_data_unique %>%
        group_by(sex_at_birth) %>%
        summarise(
          Drug = drug,
          CYP_Gene = toupper(gene),
          Window = window,
          N = n(),
          
          # ---- INHIBITORS ----
          N_Strong_Inhib = if (has_strong_inhib) sum(.data[[strong_inhib_col]] == 1, na.rm = TRUE) else NA_integer_,
          Pct_Strong_Inhib = if (has_strong_inhib) round(100 * mean(.data[[strong_inhib_col]] == 1, na.rm = TRUE), 1) else NA_real_,
          
          N_Moderate_Inhib = if (has_moderate_inhib) sum(.data[[moderate_inhib_col]] == 1, na.rm = TRUE) else NA_integer_,
          Pct_Moderate_Inhib = if (has_moderate_inhib) round(100 * mean(.data[[moderate_inhib_col]] == 1, na.rm = TRUE), 1) else NA_real_,
          
          N_Weak_Inhib = if (has_weak_inhib) sum(.data[[weak_inhib_col]] == 1, na.rm = TRUE) else NA_integer_,
          Pct_Weak_Inhib = if (has_weak_inhib) round(100 * mean(.data[[weak_inhib_col]] == 1, na.rm = TRUE), 1) else NA_real_,
          
          N_Any_Inhib = {
            any_inhib <- rep(FALSE, n())
            if (has_strong_inhib) any_inhib <- any_inhib | (.data[[strong_inhib_col]] == 1)
            if (has_moderate_inhib) any_inhib <- any_inhib | (.data[[moderate_inhib_col]] == 1)
            if (has_weak_inhib) any_inhib <- any_inhib | (.data[[weak_inhib_col]] == 1)
            if (!has_any_inhib) NA_integer_ else sum(any_inhib, na.rm = TRUE)
          },
          Pct_Any_Inhib = {
            any_inhib <- rep(FALSE, n())
            if (has_strong_inhib) any_inhib <- any_inhib | (.data[[strong_inhib_col]] == 1)
            if (has_moderate_inhib) any_inhib <- any_inhib | (.data[[moderate_inhib_col]] == 1)
            if (has_weak_inhib) any_inhib <- any_inhib | (.data[[weak_inhib_col]] == 1)
            if (!has_any_inhib) NA_real_ else round(100 * mean(any_inhib, na.rm = TRUE), 1)
          },
          
          # ---- INDUCERS ----
          N_Strong_Ind = if (has_strong_ind) sum(.data[[strong_ind_col]] == 1, na.rm = TRUE) else NA_integer_,
          Pct_Strong_Ind = if (has_strong_ind) round(100 * mean(.data[[strong_ind_col]] == 1, na.rm = TRUE), 1) else NA_real_,
          
          N_Moderate_Ind = if (has_moderate_ind) sum(.data[[moderate_ind_col]] == 1, na.rm = TRUE) else NA_integer_,
          Pct_Moderate_Ind = if (has_moderate_ind) round(100 * mean(.data[[moderate_ind_col]] == 1, na.rm = TRUE), 1) else NA_real_,
          
          N_Weak_Ind = if (has_weak_ind) sum(.data[[weak_ind_col]] == 1, na.rm = TRUE) else NA_integer_,
          Pct_Weak_Ind = if (has_weak_ind) round(100 * mean(.data[[weak_ind_col]] == 1, na.rm = TRUE), 1) else NA_real_,
          
          N_Any_Ind = {
            any_ind <- rep(FALSE, n())
            if (has_strong_ind) any_ind <- any_ind | (.data[[strong_ind_col]] == 1)
            if (has_moderate_ind) any_ind <- any_ind | (.data[[moderate_ind_col]] == 1)
            if (has_weak_ind) any_ind <- any_ind | (.data[[weak_ind_col]] == 1)
            if (!has_any_ind) NA_integer_ else sum(any_ind, na.rm = TRUE)
          },
          Pct_Any_Ind = {
            any_ind <- rep(FALSE, n())
            if (has_strong_ind) any_ind <- any_ind | (.data[[strong_ind_col]] == 1)
            if (has_moderate_ind) any_ind <- any_ind | (.data[[moderate_ind_col]] == 1)
            if (has_weak_ind) any_ind <- any_ind | (.data[[weak_ind_col]] == 1)
            if (!has_any_ind) NA_real_ else round(100 * mean(any_ind, na.rm = TRUE), 1)
          },
          
          # ---- ANY MODULATOR (inhibitor OR inducer) ----
          N_Any_Modulator = {
            any_mod <- rep(FALSE, n())
            if (has_strong_inhib) any_mod <- any_mod | (.data[[strong_inhib_col]] == 1)
            if (has_moderate_inhib) any_mod <- any_mod | (.data[[moderate_inhib_col]] == 1)
            if (has_weak_inhib) any_mod <- any_mod | (.data[[weak_inhib_col]] == 1)
            if (has_strong_ind) any_mod <- any_mod | (.data[[strong_ind_col]] == 1)
            if (has_moderate_ind) any_mod <- any_mod | (.data[[moderate_ind_col]] == 1)
            if (has_weak_ind) any_mod <- any_mod | (.data[[weak_ind_col]] == 1)
            sum(any_mod, na.rm = TRUE)
          },
          Pct_Any_Modulator = {
            any_mod <- rep(FALSE, n())
            if (has_strong_inhib) any_mod <- any_mod | (.data[[strong_inhib_col]] == 1)
            if (has_moderate_inhib) any_mod <- any_mod | (.data[[moderate_inhib_col]] == 1)
            if (has_weak_inhib) any_mod <- any_mod | (.data[[weak_inhib_col]] == 1)
            if (has_strong_ind) any_mod <- any_mod | (.data[[strong_ind_col]] == 1)
            if (has_moderate_ind) any_mod <- any_mod | (.data[[moderate_ind_col]] == 1)
            if (has_weak_ind) any_mod <- any_mod | (.data[[weak_ind_col]] == 1)
            round(100 * mean(any_mod, na.rm = TRUE), 1)
          },
          
          .groups = "drop"
        ) %>%
        relocate(sex_at_birth, .after = Window)
      
      modulator_exposure_summary_sex[[paste(drug, gene, window, sep = "_")]] <- sex_summary
    }
  }
}

# Combine all drugs
modulator_exposure_sex_df <- bind_rows(modulator_exposure_summary_sex)

# View results
print(modulator_exposure_sex_df, n = 200)

# ============================================================
# Key drugs comparison - inhibitors AND inducers by CYP gene
# ============================================================
key_drugs_comparison <- modulator_exposure_sex_df %>%
  filter(Drug %in% drugs,
         Window == "any_followup",
         !is.na(sex_at_birth)) %>%
  select(Drug, CYP_Gene, sex_at_birth, N, 
         Pct_Strong_Inhib, Pct_Moderate_Inhib, Pct_Weak_Inhib, Pct_Any_Inhib,
         Pct_Strong_Ind, Pct_Moderate_Ind, Pct_Weak_Ind, Pct_Any_Ind,
         Pct_Any_Modulator) %>%
  arrange(Drug, CYP_Gene, sex_at_birth)

print("Key drugs comparison (inhibitors and inducers by CYP gene):")
print(key_drugs_comparison, n = 100)

# ============================================================
# Female vs Male ratios for all modulator categories by CYP gene
# ============================================================
sex_ratio_all <- key_drugs_comparison %>%
  select(Drug, CYP_Gene, sex_at_birth, 
         Pct_Strong_Inhib, Pct_Moderate_Inhib, Pct_Weak_Inhib, Pct_Any_Inhib,
         Pct_Strong_Ind, Pct_Moderate_Ind, Pct_Weak_Ind, Pct_Any_Ind,
         Pct_Any_Modulator) %>%
  pivot_wider(names_from = sex_at_birth, 
              values_from = c(Pct_Strong_Inhib, Pct_Moderate_Inhib, Pct_Weak_Inhib, Pct_Any_Inhib,
                              Pct_Strong_Ind, Pct_Moderate_Ind, Pct_Weak_Ind, Pct_Any_Ind,
                              Pct_Any_Modulator)) %>%
  mutate(
    # Inhibitor ratios
    Ratio_Strong_Inhib_F_vs_M = Pct_Strong_Inhib_Female / Pct_Strong_Inhib_Male,
    Ratio_Moderate_Inhib_F_vs_M = Pct_Moderate_Inhib_Female / Pct_Moderate_Inhib_Male,
    Ratio_Weak_Inhib_F_vs_M = Pct_Weak_Inhib_Female / Pct_Weak_Inhib_Male,
    Ratio_Any_Inhib_F_vs_M = Pct_Any_Inhib_Female / Pct_Any_Inhib_Male,
    # Inducer ratios
    Ratio_Strong_Ind_F_vs_M = Pct_Strong_Ind_Female / Pct_Strong_Ind_Male,
    Ratio_Moderate_Ind_F_vs_M = Pct_Moderate_Ind_Female / Pct_Moderate_Ind_Male,
    Ratio_Weak_Ind_F_vs_M = Pct_Weak_Ind_Female / Pct_Weak_Ind_Male,
    Ratio_Any_Ind_F_vs_M = Pct_Any_Ind_Female / Pct_Any_Ind_Male,
    # Overall modulator ratio
    Ratio_Any_Modulator_F_vs_M = Pct_Any_Modulator_Female / Pct_Any_Modulator_Male
  )

# Simplified view - just the ratios by gene
print("\nFemale vs Male ratios (>1 = more common in females):")

print("\nInhibitor ratios by CYP gene:")
print(sex_ratio_all %>% select(Drug, CYP_Gene, starts_with("Ratio") & contains("Inhib")), n = 100)

print("\nInducer ratios by CYP gene:")
print(sex_ratio_all %>% select(Drug, CYP_Gene, starts_with("Ratio") & contains("Ind") & !contains("Inhib")), n = 100)

print("\nOverall modulator ratio by CYP gene:")
print(sex_ratio_all %>% select(Drug, CYP_Gene, Ratio_Any_Modulator_F_vs_M), n = 100)

# ============================================================
# Statistical tests for sex differences (chi-squared) by CYP gene
# ============================================================
print("\n--- Chi-squared tests for sex differences in modulator exposure ---")

for (drug_name in drugs) {
  for (gene_name in c("CYP2D6", "CYP2C19", "CYP2B6")) {
    
    drug_subset <- modulator_exposure_sex_df %>%
      filter(Drug == drug_name, CYP_Gene == gene_name, 
             Window == "any_followup", !is.na(sex_at_birth))
    
    if (nrow(drug_subset) < 2) next
    
    cat(paste0("\n", drug_name, " - ", gene_name, ":\n"))
    
    # Test each modulator category
    for (mod_type in c("Any_Inhib", "Strong_Inhib", "Moderate_Inhib", "Weak_Inhib",
                       "Any_Ind", "Strong_Ind", "Moderate_Ind", "Weak_Ind",
                       "Any_Modulator")) {
      
      n_col <- paste0("N_", mod_type)
      pct_col <- paste0("Pct_", mod_type)
      if (!n_col %in% names(drug_subset)) next
      if (all(is.na(drug_subset[[n_col]]))) next
      
      exposed <- drug_subset[[n_col]]
      totals <- drug_subset$N
      unexposed <- totals - exposed
      
      if (any(is.na(exposed)) || any(exposed < 0)) next
      
      mat <- matrix(c(exposed, unexposed), nrow = 2, byrow = FALSE)
      
      tryCatch({
        test <- chisq.test(mat)
        pct_f <- drug_subset[[pct_col]][drug_subset$sex_at_birth == "Female"]
        pct_m <- drug_subset[[pct_col]][drug_subset$sex_at_birth == "Male"]
        cat(sprintf("  %s: F=%.1f%% vs M=%.1f%%, chi2=%.2f, p=%.2e\n",
                    mod_type, pct_f, pct_m, test$statistic, test$p.value))
      }, error = function(e) {
        cat(sprintf("  %s: test failed (%s)\n", mod_type, e$message))
      })
    }
  }
}

format_sigfig <- function(x, sf = 2) {
  if (is.na(x)) return(NA_character_)
  if (x == 0)   return("0")
  rounded <- signif(x, sf)
  dp <- max(0, sf - 1 - floor(log10(abs(rounded))))
  formatC(rounded, format = "f", digits = dp)
}


# ============================================================
# Combine everything into one summary dataframe
# ============================================================

# Pull N_exposed columns through alongside the Pct columns
key_drugs_comparison_full <- modulator_exposure_sex_df %>%
  filter(Drug %in% drugs,
         Window == "any_followup",
         !is.na(sex_at_birth)) %>%
  select(Drug, CYP_Gene, sex_at_birth, N,
         N_Strong_Inhib, Pct_Strong_Inhib,
         N_Moderate_Inhib, Pct_Moderate_Inhib,
         N_Weak_Inhib, Pct_Weak_Inhib,
         N_Any_Inhib, Pct_Any_Inhib,
         N_Strong_Ind, Pct_Strong_Ind,
         N_Moderate_Ind, Pct_Moderate_Ind,
         N_Weak_Ind, Pct_Weak_Ind,
         N_Any_Ind, Pct_Any_Ind,
         N_Any_Modulator, Pct_Any_Modulator) %>%
  arrange(Drug, CYP_Gene, sex_at_birth)

# 1. Reshape to wide
wide_by_sex <- key_drugs_comparison_full %>%
  pivot_wider(
    id_cols = c(Drug, CYP_Gene),
    names_from = sex_at_birth,
    values_from = c(N,
                    N_Strong_Inhib, Pct_Strong_Inhib,
                    N_Moderate_Inhib, Pct_Moderate_Inhib,
                    N_Weak_Inhib, Pct_Weak_Inhib,
                    N_Any_Inhib, Pct_Any_Inhib,
                    N_Strong_Ind, Pct_Strong_Ind,
                    N_Moderate_Ind, Pct_Moderate_Ind,
                    N_Weak_Ind, Pct_Weak_Ind,
                    N_Any_Ind, Pct_Any_Ind,
                    N_Any_Modulator, Pct_Any_Modulator)
  )

# 2. Compute ratios
wide_by_sex <- wide_by_sex %>%
  mutate(
    Ratio_Strong_Inhib_F_vs_M   = Pct_Strong_Inhib_Female   / Pct_Strong_Inhib_Male,
    Ratio_Moderate_Inhib_F_vs_M = Pct_Moderate_Inhib_Female / Pct_Moderate_Inhib_Male,
    Ratio_Weak_Inhib_F_vs_M     = Pct_Weak_Inhib_Female     / Pct_Weak_Inhib_Male,
    Ratio_Any_Inhib_F_vs_M      = Pct_Any_Inhib_Female      / Pct_Any_Inhib_Male,
    Ratio_Strong_Ind_F_vs_M     = Pct_Strong_Ind_Female     / Pct_Strong_Ind_Male,
    Ratio_Moderate_Ind_F_vs_M   = Pct_Moderate_Ind_Female   / Pct_Moderate_Ind_Male,
    Ratio_Weak_Ind_F_vs_M       = Pct_Weak_Ind_Female       / Pct_Weak_Ind_Male,
    Ratio_Any_Ind_F_vs_M        = Pct_Any_Ind_Female        / Pct_Any_Ind_Male,
    Ratio_Any_Modulator_F_vs_M  = Pct_Any_Modulator_Female  / Pct_Any_Modulator_Male
  )

# 3. Run chi-squared tests and collect p-values
chisq_results <- list()
for (i in seq_len(nrow(wide_by_sex))) {
  row <- wide_by_sex[i, ]
  drug_name <- row$Drug
  gene_name <- row$CYP_Gene

  drug_subset <- modulator_exposure_sex_df %>%
    filter(Drug == drug_name, CYP_Gene == gene_name,
           Window == "any_followup", !is.na(sex_at_birth))

  if (nrow(drug_subset) < 2) next

  row_pvals <- tibble(Drug = drug_name, CYP_Gene = gene_name)

  for (mod_type in c("Any_Inhib", "Strong_Inhib", "Moderate_Inhib", "Weak_Inhib",
                     "Any_Ind", "Strong_Ind", "Moderate_Ind", "Weak_Ind",
                     "Any_Modulator")) {

    n_col <- paste0("N_", mod_type)
    p_col <- paste0("P_", mod_type)

    if (!n_col %in% names(drug_subset) || all(is.na(drug_subset[[n_col]]))) {
      row_pvals[[p_col]] <- NA_real_; next
    }

    exposed   <- drug_subset[[n_col]]
    totals    <- drug_subset$N
    unexposed <- totals - exposed

    if (any(is.na(exposed)) || any(exposed < 0)) {
      row_pvals[[p_col]] <- NA_real_; next
    }

    mat <- matrix(c(exposed, unexposed), nrow = 2, byrow = FALSE)

    tryCatch({
      test <- chisq.test(mat)
      row_pvals[[p_col]] <- test$p.value
    }, error = function(e) {
      row_pvals[[p_col]] <<- NA_real_
    })
  }

  chisq_results[[paste(drug_name, gene_name, sep = "_")]] <- row_pvals
}
chisq_df <- bind_rows(chisq_results)

# 4. Join
combined_summary <- wide_by_sex %>%
  left_join(chisq_df, by = c("Drug", "CYP_Gene")) %>%
  arrange(Drug, CYP_Gene)

# 5. Long format - one row per Drug x CYP_Gene x Modulator_Type,
#    carrying through the exposed Ns so we can filter on them
mod_types <- c("Strong_Inhib", "Moderate_Inhib", "Weak_Inhib", "Any_Inhib",
               "Strong_Ind",   "Moderate_Ind",   "Weak_Ind",   "Any_Ind",
               "Any_Modulator")

combined_long <- map_dfr(mod_types, function(mod) {
  combined_summary %>%
    select(Drug, CYP_Gene, N_Female, N_Male,
           matches(paste0("^N_",     mod, "_(Female|Male)$")),
           matches(paste0("^Pct_",   mod, "_(Female|Male)$")),
           matches(paste0("^Ratio_", mod)),
           matches(paste0("^P_",     mod))) %>%
    mutate(Modulator_Type = mod) %>%
    rename_with(~ "N_exposed_Female",  matches(paste0("^N_",   mod, "_Female$"))) %>%
    rename_with(~ "N_exposed_Male",    matches(paste0("^N_",   mod, "_Male$"))) %>%
    rename_with(~ "Pct_Female",        matches(paste0("^Pct_", mod, "_Female$"))) %>%
    rename_with(~ "Pct_Male",          matches(paste0("^Pct_", mod, "_Male$"))) %>%
    rename_with(~ "Ratio_F_vs_M",      matches(paste0("^Ratio_", mod))) %>%
    rename_with(~ "P_value",           matches(paste0("^P_", mod)))
})

combined_long <- as.data.frame(combined_long) %>%
  select(Drug, CYP_Gene, Modulator_Type,
         N_Female, N_Male, N_exposed_Female, N_exposed_Male,
         Pct_Female, Pct_Male, Ratio_F_vs_M, P_value) %>%
  filter(!is.na(Pct_Female))

# Internal numeric version (keep inside the Workbench)
combined_long_numeric <- combined_long

# AoU-compliant version: drop rows where either sex has <20 exposed
combined_long_suppressed <- combined_long %>%
  filter(
    !is.na(N_exposed_Female) & !is.na(N_exposed_Male),
    N_exposed_Female >= 20,
    N_exposed_Male   >= 20
  ) %>%
  mutate(
    Pct_Female   = vapply(Pct_Female,   format_sigfig, character(1), sf = 2),
    Pct_Male     = vapply(Pct_Male,     format_sigfig, character(1), sf = 2),
    Ratio_F_vs_M = vapply(Ratio_F_vs_M, format_sigfig, character(1), sf = 2)
  )

print(combined_long_suppressed)

In [ ]:
wb <- loadWorkbook("/home/jupyter/workspaces/cypphenoconversion/data/All_Results_CYP.xlsx")

removeWorksheet(wb, "SexStratified_Mod_Prevalence")
addWorksheet(wb, "SexStratified_Mod_Prevalence")
writeData(wb, "SexStratified_Mod_Prevalence", combined_long_suppressed)

saveWorkbook(wb, "/home/jupyter/workspaces/cypphenoconversion/data/All_Results_CYP.xlsx", overwrite=TRUE)